# Differential interconnects, crosstalk, and time-domain reflectometry with Palace

This tutorial addresses [gsim #274](https://github.com/gdsfactory/gsim/issues/274).
We model **two IHP SG13G2 signal traces over a common return plane**, then cut an
asymmetric slot in that plane. Does the slot increase reflections, couple energy
between the signals, or convert differential excitation into common mode? Can we
locate it from the reflected waveform?

Palace performs the **electromagnetic (EM)** frequency-domain solve. Python and
scikit-rf perform network conversion and **time-domain reflectometry (TDR)**: the
reflected response to an incident voltage step. All helper functions live here.

The **process design kit (PDK)** supplies process layers. Geometry helpers accept
layer selections, simulation helpers accept a resolved stack, and analysis helpers
accept networks and port names. Only the IHP setup section knows IHP layer names.
A small synthetic PDK check demonstrates this separation.

**Execution:** run the notebook in order. The default `EXECUTION = "recorded"` replays the completed local experiments if
the accompanying run directories are present. The repository strips cell outputs; the numerical discussion below and exported
figures in `docs/assets/plots/palace-si-*.png` retain selected results. Raw run
archives are not bundled; use local/cloud execution to regenerate them. `"preview"` runs analytic checks and builds layouts without
meshing or submitting jobs.
Choose `"local"` or `"cloud"` to generate EM data, or `"load"` to analyze the exact
saved experiment. The convergence study is separately enabled. Missing runs are
reported as **incomplete**, never replaced by analytic curves.

This notebook requires gsim's PDK-authoritative stack API (PR #360; base commit
`33fa363b7499a68c2818f72289395739fd76ce55`). No helper scripts or measurement files
are required. Local runs require a Palace installation; cloud runs require a
configured GDSFactory+ account. Full convergence runs can be expensive.

## 1. Environment and reproducibility

Use a kernel with this gsim checkout installed. Dependencies used while developing
this tutorial are `ihp-gdsfactory==0.2.8`, `scikit-rf==2.1.0`, `gdsfactory==9.40.2`,
NumPy, SciPy, pandas, and Matplotlib. For example, install the first two with
`uv pip install 'ihp-gdsfactory==0.2.8' 'scikit-rf==2.1.0'` in that environment.

The IHP release is pinned to **v0.2.8**, whose source commit is
[`d119d9ce0e9ad3092654a1cc9e1d0e18c1d6618f`](https://github.com/gdsfactory/IHP/tree/d119d9ce0e9ad3092654a1cc9e1d0e18c1d6618f).
Package versions, available installation provenance, geometry, resolved stack,
material overrides, port planes, solver settings, and data hashes are saved with
completed runs. Record the Palace runtime version in the controls before a solve;
retain the solver's logs alongside the generated mesh and configuration.

Coordinates are in **micrometers (µm)**, frequencies in **hertz (Hz)**, and times
in **seconds (s)** unless an axis explicitly says otherwise.

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import logging
import shutil
import subprocess
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skrf as rf
from IPython.display import display as show
from matplotlib.patches import Rectangle

from gsim.common.pdk import resolve_layer_stack
from gsim.palace import DrivenSim
from gsim.palace.stack import build_palace_layer_stack

EXECUTION = "recorded"  # recorded | preview | load | local | cloud
RUN_CONVERGENCE = False
OUTPUT = (
    Path("..") if Path.cwd().name == "nbs" else Path()
) / "palace-signal-integrity-data"
PALACE_EXECUTABLE = str(
    Path.home() / ".cache/palacetoolkit/runtime/palace-cpu-v0.18.0/bin/palace"
)  # e.g. "/opt/palace/bin/palace"; None uses runtime discovery
PALACE_VERSION = "v0.18.0 (palacetoolkit CPU runtime; schema 1-6-0)"
LOCAL_PROCESSES = 4
PORTS = ("p1", "p2", "p3", "p4")
PAIRS = (("p1", "p2"), ("p3", "p4"))
Z_SE = 50.0

if EXECUTION not in {"recorded", "preview", "load", "local", "cloud"}:
    raise ValueError("Unknown execution mode")
if EXECUTION in {"local", "cloud"} and PALACE_VERSION == "UNRECORDED":
    raise ValueError("Record PALACE_VERSION before launching the experiment")

VERSIONS = {
    p: metadata.version(p)
    for p in ("gsim", "ihp-gdsfactory", "scikit-rf", "gdsfactory", "numpy", "scipy")
}
if VERSIONS["ihp-gdsfactory"] != "0.2.8":
    raise RuntimeError("This device example is pinned to ihp-gdsfactory==0.2.8")
if VERSIONS["scikit-rf"] != "2.1.0":
    raise RuntimeError("Use scikit-rf==2.1.0 to reproduce this tutorial")
try:
    GSIM_REVISION = subprocess.check_output(  # noqa: S603 -- fixed read-only git arguments
        [shutil.which("git") or "/usr/bin/git", "rev-parse", "HEAD"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GSIM_REVISION = "unavailable"
IHP_INSTALL = metadata.distribution("ihp-gdsfactory").read_text("direct_url.json")
show(pd.Series(VERSIONS, name="Installed version").to_frame())

## 2. Four terminals, two complementary measurements

```text
               near plane x = 0             far plane x = L
p1 (+)  ------------------- signal A ------------------- p3 (+)
p2 (-)  ------------------- signal B ------------------- p4 (-)
                 a common connected ground plane below
```

Each port voltage is measured from its signal to ground. The ground is a shared
reference, not a fifth signal port. The ground slot is enclosed by metal, so it
does not split the return into floating conductors. Port sheets and end geometry
are identical in all comparisons. We retain the launches in the reference planes;
there is no hidden de-embedding (removal of fixtures or launches).

For single-ended excitation at p1 with all other ports matched to 50 Ω:

| Quantity | Meaning | S-parameter |
|---|---|---|
| Reflection | Returns to p1 | $S_{11}$ |
| Transmission | Arrives on A at the far end | $S_{31}$ |
| NEXT: near-end crosstalk | Arrives on B at the near end | $S_{21}$ |
| FEXT: far-end crosstalk | Arrives on B at the far end | $S_{41}$ |

We use $b_i=\sum_j S_{ij}a_j$: row = response, column = excitation. Four independent
excitations are needed for the full matrix, especially after breaking symmetry.
This is one differential pair. Coupling between **two differential pairs** would
require eight signal ports.

For differential/common-mode analysis, pair (p1,p2) and (p3,p4). Differential voltage
is $V_d=V_A-V_B$; common voltage is $V_c=(V_A+V_B)/2$. Corresponding currents are
$I_d=(I_A-I_B)/2$ and $I_c=I_A+I_B$. Thus the references are 100 Ω differential and
25 Ω common for 50 Ω per terminal. With power-normalized waves,

$$S_{mm}=M S_{se}M^T,\qquad
M=\frac{1}{\sqrt2}\begin{bmatrix}
1&-1&0&0\\0&0&1&-1\\1&1&0&0\\0&0&1&1
\end{bmatrix}.$$

The mixed-mode order is $(d_1,d_2,c_1,c_2)$, and
$S_{mm}=\left[\begin{smallmatrix}S_{dd}&S_{dc}\\S_{cd}&S_{cc}\end{smallmatrix}\right]$.
For example, $S_{cd}$ measures common response to differential excitation. The
simple orthogonal formula assumes equal, real single-ended references; the helper
below explicitly enforces this condition.

The saved Palace files use the internal order `(p1, p3, p2, p4)` because the
geometry creates the two ends of each trace together. Always read
`port_information.json`: the analysis reorders by name before applying the table
and mixed-mode transformation above. Raw numeric CSV indices are not the tutorial
indices until that reordering has been performed.

In [ ]:
M = np.array(
    [[1, -1, 0, 0], [0, 0, 1, -1], [1, 1, 0, 0], [0, 0, 1, 1]], dtype=float
) / np.sqrt(2)


def check_network(net):
    if net.nports != 4 or not np.isfinite(net.s).all():
        raise ValueError("Expected a finite, complete four-port network")
    if len(net.f) < 2 or not np.all(np.diff(net.f) > 0):
        raise ValueError("Frequency must be strictly increasing")
    if net.port_names is None or len(set(net.port_names)) != 4:
        raise ValueError("Four unique port names are required")


def mixed_mode(net, pairs=PAIRS, z_se=Z_SE):
    check_network(net)
    names = [name for pair in pairs for name in pair]
    if len(pairs) != 2 or any(len(pair) != 2 for pair in pairs):
        raise ValueError("Supply two ordered (+, -) pairs")
    if len(set(names)) != 4 or set(names) != set(net.port_names):
        raise ValueError("Pair names must cover each port exactly once")
    if not np.allclose(net.z0, z_se, rtol=0, atol=1e-12):
        raise ValueError("Renormalize explicitly to equal real z_se first")
    indices = [net.port_names.index(name) for name in names]
    ordered = rf.Network(
        f=net.f, s=net.s[:, indices][:, :, indices], z0=z_se, f_unit="Hz", name=net.name
    )
    ordered.port_names = names
    mm = ordered.copy()
    mm.se2gmm(p=2)
    mm.port_names = ["d_near", "d_far", "c_near", "c_far"]
    return ordered, mm


def complete_network(parameters):
    # Check original entries BEFORE to_skrf() can fill missing reciprocal entries
    # or leave unobserved entries at zero. Preserve the stored impedance.
    expected = {(a, b) for a in PORTS for b in PORTS}
    if set(parameters.keys()) != expected:
        raise ValueError(
            f"Incomplete/unexpected matrix: missing {expected - set(parameters.keys())}"
        )
    net = parameters.to_skrf()
    check_network(net)
    if not np.allclose(net.z0, Z_SE):
        raise ValueError("Solver reference differs from the declared 50 ohm setup")
    return mixed_mode(net)[0]


# Independent algebraic oracle, including arbitrary port ordering and inverse.
rng = np.random.default_rng(274)
s = (rng.normal(size=(7, 4, 4)) + 1j * rng.normal(size=(7, 4, 4))) * 0.05
trial = rf.Network(f=np.linspace(1e9, 7e9, 7), s=s, z0=Z_SE, f_unit="Hz")
trial.port_names = list(PORTS)
for perm in ([0, 1, 2, 3], [2, 0, 3, 1], [1, 0, 3, 2]):
    shuffled = rf.Network(f=trial.f, s=s[:, perm][:, :, perm], z0=Z_SE, f_unit="Hz")
    shuffled.port_names = [PORTS[i] for i in perm]
    ordered, mm = mixed_mode(shuffled)
    np.testing.assert_allclose(mm.s, M @ s @ M.T, atol=1e-12, rtol=0)
    back = mm.copy()
    back.gmm2se(p=2)
    np.testing.assert_allclose(back.s, ordered.s, atol=1e-12, rtol=0)
    restore = [ordered.port_names.index(name) for name in shuffled.port_names]
    np.testing.assert_allclose(
        back.s[:, restore][:, :, restore], shuffled.s, atol=1e-12, rtol=0
    )
    np.testing.assert_allclose(back.z0, Z_SE, atol=1e-12)
print("Mixed-mode formula and inverse agree to 1e-12, including reordered ports.")

## 3. Understand TDR on a known line before interpreting EM data

For an ideal line of characteristic impedance $Z_c$, one-way delay $\tau$, and
reference impedance $Z_r$, define $\theta=j2\pi f\tau$. Its transmission matrix is

$$A=D=\cosh\theta,\quad B=Z_c\sinh\theta,\quad C=\sinh\theta/Z_c.$$

With $q=A+B/Z_r+C Z_r+D$,
$S_{11}=(A+B/Z_r-C Z_r-D)/q$ and $S_{21}=2/q$.
We use this independent analytic model to check the time transform. A mismatched
line is necessary: a matched line alone would not test impedance recovery.

The transform needs a **uniform frequency grid including DC (zero frequency)**.
For EM data beginning at $\Delta f$, we add a real DC reflection coefficient by
linear extrapolation of the lowest two real samples; the imaginary DC value is
zero for a real impulse response. This is an assumption about missing low-frequency
behavior, not a Palace DC solve. We also compare a constant extrapolation.

A Hamming window reduces ringing from the frequency cutoff but broadens events.
The sampling interval is approximately $1/(2f_{max})$ and the periodic time span is
$1/\Delta f$. scikit-rf centers that interval around zero. Zero padding gives a
smoother display, not additional bandwidth. We use $2/f_{max}$ as a conservative
working event-separation scale and verify errors on analytic examples.

For reflected step amplitude $\rho(t)$,

$$Z_{TDR}(t)=Z_r\frac{1+\rho(t)}{1-\rho(t)}.$$

This is an **apparent impedance** for our lossy coupled device. It need not equal
local characteristic impedance, especially after multiple reflections arrive.
Other mixed-mode ports remain terminated at their reference impedances.

The 250 MHz frequency step was selected after checking DC extrapolation on the
analytic line. A 500 MHz step gave about 3.1% impedance error with the same linear
DC policy; the finer grid meets the 2% analytic criterion without changing it.

In [ ]:
def line_network(f, zc, delay, zref):
    theta = 2j * np.pi * np.asarray(f) * delay
    a, b, c = np.cosh(theta), zc * np.sinh(theta), np.sinh(theta) / zc
    denominator = 2 * a + b / zref + c * zref
    reflection = (b / zref - c * zref) / denominator
    transmission = 2 / denominator
    s = np.empty((len(f), 2, 2), complex)
    s[:, 0, 0] = s[:, 1, 1] = reflection
    s[:, 0, 1] = s[:, 1, 0] = transmission
    return rf.Network(f=f, s=s, z0=zref, f_unit="Hz")


def tdr(f, reflection, zref, window="hamming", dc="linear", pad=0):
    f, reflection = np.asarray(f, float), np.asarray(reflection, complex)
    if f.ndim != 1 or len(f) < 3 or reflection.shape != f.shape:
        raise ValueError("Need at least three scalar reflection samples")
    if not np.isfinite(f).all() or not np.isfinite(reflection).all():
        raise ValueError("Nonfinite TDR input")
    df = f[1] - f[0]
    if df <= 0 or not np.allclose(np.diff(f), df, rtol=1e-8, atol=1):
        raise ValueError("TDR requires a uniform increasing frequency grid")
    if zref <= 0 or dc not in {"linear", "constant"}:
        raise ValueError("Use a positive reference and linear/constant DC policy")
    if np.isclose(f[0], df, rtol=1e-8, atol=1):
        value = (
            2 * reflection[0].real - reflection[1].real
            if dc == "linear"
            else reflection[0].real
        )
        reflection = np.r_[value, reflection]
        f = np.r_[0.0, f]
    elif f[0] != 0:
        raise ValueError("Start at DC or exactly one frequency step above DC")
    if abs(reflection[0].imag) > 1e-10:
        raise ValueError("DC reflection must be real")
    oneport = rf.Network(f=f, s=reflection[:, None, None], z0=zref, f_unit="Hz")
    t, rho = oneport.step_response(window=window, pad=pad)
    denominator = 1 - rho
    impedance = np.full(rho.shape, np.nan)
    np.divide(
        zref * (1 + rho), denominator, out=impedance, where=np.abs(denominator) > 1e-6
    )
    return t, rho, impedance


F_ANALYTIC = np.arange(801) * 0.25e9
bandwidth = F_ANALYTIC[-1]
resolution = 2 / bandwidth
zc, delay, zref = 130.0, 80e-12, 100.0
uniform = line_network(F_ANALYTIC, zc, delay, zref)
t, rho, z = tdr(F_ANALYTIC, uniform.s[:, 0, 0], zref)
plateau = (t > 4 * resolution) & (t < 2 * delay - 4 * resolution)
assert plateau.any(), "Increase bandwidth or analytic line length"
z_estimate = np.median(z[plateau])
# First transmitted impulse arrives at the one-way delay.
t_imp, impulse = uniform.s21.impulse_response(window="hamming", bandpass=False)
tau_estimate = t_imp[np.argmax(np.abs(impulse))]
assert abs(z_estimate / zc - 1) < 0.02
assert abs(tau_estimate - delay) < resolution

# A matched 100-ohm launch line ending in 150 ohms gives a delayed step.
# Its exact reflection is Gamma * exp(-j*2*pi*f*2*tau).
interface_delay = 40e-12
interface_gamma = (150 - zref) / (150 + zref)
interface_s11 = interface_gamma * np.exp(-4j * np.pi * F_ANALYTIC * interface_delay)
t_gap, r_gap, _ = tdr(F_ANALYTIC, interface_s11, zref)
arrival = t_gap[np.argmin(np.abs(r_gap - interface_gamma / 2))]
assert abs(arrival - 2 * interface_delay) < resolution
assert np.median(r_gap[(t_gap > 120e-12) & (t_gap < 200e-12)]) > 0

# Exercise the DC reconstruction used for real runs, with known DC withheld.
t_ext, r_ext, z_ext = tdr(F_ANALYTIC[1:], uniform.s[1:, 0, 0], zref)
assert abs(np.median(z_ext[plateau]) / zc - 1) < 0.02
analytic_report = pd.DataFrame(
    [
        {
            "check": "uniform impedance",
            "expected": zc,
            "obtained": z_estimate,
            "unit": "ohm",
        },
        {
            "check": "one-way delay",
            "expected": delay * 1e12,
            "obtained": tau_estimate * 1e12,
            "unit": "ps",
        },
        {
            "check": "interface round trip",
            "expected": 2 * interface_delay * 1e12,
            "obtained": arrival * 1e12,
            "unit": "ps",
        },
    ]
)
show(analytic_report)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(t * 1e12, z, label="Analytic uniform line")
axes[0].axhline(zc, color="k", ls=":", label="Known Zc")
axes[0].set(
    xlim=(-20, 220),
    ylim=(80, 150),
    xlabel="Time (ps)",
    ylabel="Apparent impedance (ohm)",
)
axes[1].plot(t_gap * 1e12, r_gap, label="Analytic delayed interface")
axes[1].axvline(2 * interface_delay * 1e12, color="k", ls=":", label="Known round trip")
axes[1].set(
    xlim=(-20, 220), xlabel="Time (ps)", ylabel="Reflected step / incident step"
)
for ax in axes:
    ax.grid()
    ax.legend()
fig.tight_layout()

## 4. A generic geometry builder

Our two traces sit above a rectangular ground plane. We remove a rectangular
**interior** slot beneath signal A, leaving metal paths around every side. In the
intact layout the exchange A ↔ B is a symmetry. The offset slot breaks this symmetry;
a centered slot would not necessarily create differential/common-mode conversion.

Dimensions below are initial experiment choices, not an optimized or foundry-qualified
interconnect. The 6 mm line and 200 GHz upper frequency are chosen to separate
launch, slot, and far-end events. A 120 µm slot may appear as one broadened event:
we seek its location, not resolution of both edges.

In [ ]:
@dataclass(frozen=True)
class Geometry:
    length: float = 6000.0
    width: float = 14.0
    separation: float = 40.0  # signal center-to-center distance
    ground_width: float = 240.0
    gap_x: float = 2400.0  # slot center, measured from near reference plane
    gap_length: float = 120.0
    gap_y: float = 20.0
    gap_width: float = 36.0


GEO = Geometry()


def make_interconnect(g, signal_layer, return_layer, gap=False):
    if min(g.length, g.width, g.separation, g.ground_width) <= 0:
        raise ValueError("Geometry dimensions must be positive")
    if g.separation <= g.width or g.ground_width <= g.separation + g.width:
        raise ValueError("Signals must be disjoint and fit above the ground")
    c = gf.Component()
    for y, near, far in [
        (g.separation / 2, "p1", "p3"),
        (-g.separation / 2, "p2", "p4"),
    ]:
        c.add_polygon(
            [
                (0, y - g.width / 2),
                (g.length, y - g.width / 2),
                (g.length, y + g.width / 2),
                (0, y + g.width / 2),
            ],
            layer=signal_layer,
        )
        c.add_port(
            name=near,
            center=(0, y),
            width=g.width,
            orientation=180,
            layer=signal_layer,
            port_type="electrical",
        )
        c.add_port(
            name=far,
            center=(g.length, y),
            width=g.width,
            orientation=0,
            layer=signal_layer,
            port_type="electrical",
        )
    dbu = c.kcl.dbu

    def box(x0, y0, x1, y1):
        return gf.kdb.Box(*(round(v / dbu) for v in (x0, y0, x1, y1)))

    ground = gf.kdb.Region(box(0, -g.ground_width / 2, g.length, g.ground_width / 2))
    if gap:
        x0, x1 = g.gap_x - g.gap_length / 2, g.gap_x + g.gap_length / 2
        y0, y1 = g.gap_y - g.gap_width / 2, g.gap_y + g.gap_width / 2
        if not (
            0 < x0 < x1 < g.length
            and -g.ground_width / 2 < y0 < y1 < g.ground_width / 2
        ):
            raise ValueError("Slot must be strictly inside the ground plane")
        ground -= gf.kdb.Region(box(x0, y0, x1, y1))
    ground.merge()
    if ground.count() != 1:
        raise ValueError("Ground must remain one connected polygon")
    c.add_polygon(ground, layer=return_layer)
    return c


def show_layout(g):
    fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
    for ax, gap, name in zip(axes, [False, True], ["Intact", "Disrupted"], strict=True):
        ax.add_patch(
            Rectangle(
                (0, -g.ground_width / 2),
                g.length,
                g.ground_width,
                color="lightgray",
                label="Return plane (lower layer)",
            )
        )
        if gap:
            ax.add_patch(
                Rectangle(
                    (g.gap_x - g.gap_length / 2, g.gap_y - g.gap_width / 2),
                    g.gap_length,
                    g.gap_width,
                    color="white",
                    hatch="//",
                )
            )
        for y, near, far in [
            (g.separation / 2, "p1", "p3"),
            (-g.separation / 2, "p2", "p4"),
        ]:
            ax.add_patch(
                Rectangle(
                    (0, y - g.width / 2),
                    g.length,
                    g.width,
                    color="tab:blue",
                    alpha=0.65,
                )
            )
            ax.text(-60, y, near, ha="right", va="center")
            ax.text(g.length + 60, y, far, va="center")
        ax.set(
            xlim=(-250, g.length + 250), ylim=(-150, 150), ylabel="y (um)", title=name
        )
    axes[-1].set_xlabel("x (um); transverse scale enlarged for readability")
    fig.tight_layout()


show_layout(GEO)

## 5. Select IHP layers and declare the physical model

This is the PDK-specific section. The generic builder uses the resolved GDS layer
numbers; the solver uses the resolved vertical extents and materials.

The PDK-authoritative branch's IHP overlay supplies missing bulk dielectric regions
and via roles. Here we explicitly replace its `ild` background with oxide from
$z=0$ to the declared passivation, and add **180 µm of silicon below $z=0$**.
Passivation comes from the overlay/PDK. No new via geometry is needed for these
interlayer port sheets.

We use the registry's metal and dielectric properties and explicitly set silicon
$\epsilon_r=11.7$, conductivity $\sigma=10\ \mathrm{S/m}$, and dielectric loss tangent
zero (conductivity still causes loss). These are **tutorial modeling assumptions**,
not certified IHP RF substrate parameters. The 180 µm truncation and outer absorbing
boundary require sensitivity checks for quantitative substrate-mediated coupling.
Inspect the tables before drawing process conclusions.

Aluminum uses the registry's standard conductivity, $3.77\times10^7$ S/m,
through finite-thickness conductivity boundaries. Its attached material-model
metadata lists a frequency-validity ceiling of 100 GHz. This tutorial explicitly
continues the **constant-conductivity assumption to 200 GHz** for the TDR
bandwidth study. Numerical mesh convergence does not validate that material
extrapolation or provide measured foundry data.


In [ ]:
from ihp import PDK

PDK.activate()
IHP_REVISION = "d119d9ce0e9ad3092654a1cc9e1d0e18c1d6618f"
initial_stack = resolve_layer_stack(pdk=PDK)
SIGNAL_KEY, RETURN_KEY = "topmetal2", "metal1"
SIGNAL_LAYER = initial_stack.layers[SIGNAL_KEY].gds_layer
RETURN_LAYER = initial_stack.layers[RETURN_KEY].gds_layer
BULK = [
    {
        "name": "ild",
        "material": "sio2",
        "zmin": 0.0,
        "zmax": initial_stack.layers["passiv"].zmin,
    },
    {"name": "substrate", "material": "silicon", "zmin": -180.0, "zmax": 0.0},
]
MATERIAL_OVERRIDES = {
    "silicon": {
        "material_type": "semiconductor",
        "permittivity": 11.7,
        "conductivity": 10.0,
        "loss_tangent": 0.0,
    },
}
components = {
    name: make_interconnect(GEO, SIGNAL_LAYER, RETURN_LAYER, gap)
    for name, gap in [("intact", False), ("disrupted", True)]
}
resolved = resolve_layer_stack(components["intact"], pdk=PDK, dielectrics=BULK)
show(pd.DataFrame([asdict(resolved.layers[k]) for k in (SIGNAL_KEY, RETURN_KEY)]))
show(pd.DataFrame([asdict(d) for d in resolved.dielectrics]))
show(pd.DataFrame.from_dict(dict(resolved.materials), orient="index"))
show(pd.DataFrame.from_dict(MATERIAL_OVERRIDES, orient="index"))
print("Overlay:", resolved.overlay)
print("Reference planes: x=0 and x=", GEO.length, "um; no de-embedding")

## 6. Configure the solver through PDK-independent inputs

We use **lumped terminal ports**: vertical sheets from the return layer to each
signal, referenced to 50 Ω. Set `excited=True` on all four; Palace assigns separate
excitation IDs. We inspect the generated JSON before running.

The executed dense profile uses first-order finite elements with an 8 µm refined
mesh size. The original second-order, 6 µm profile is retained as `FULL_SETTINGS`,
but its full-band direct solve exceeded the practical memory budget. The recorded
results are exploratory; the final section quantifies the failed convergence checks. Finite-thickness metal
geometry is retained (`planar_conductors=False`); the generated Palace conductivity
boundaries and material tables should be inspected with the mesh. The airbox and
substrate are part of the physical model, not merely display padding.

The AWS CPW example motivates a launch-refinement check and comparison of adaptive
frequency sweeps with direct solves. Weak crosstalk may need much tighter accuracy
than through transmission. Here `adaptive_tol=1e-5` is a starting choice, not a
certified bound on any S-parameter. Direct anchor solves use `adaptive_tol=0`.

Local execution uses the MUMPS direct solver with relative linear tolerance
`1e-8`. A pilot benchmark found this substantially faster than the default
iterative preconditioner for these long thin structures. This changes the linear
algebra, not the physical geometry; solver selection and tolerance are recorded
in the experiment settings. Each completed run retains `solver.log`.

Future local runs allow up to five linear correction iterations. The archived
dense runs used the original one-iteration MUMPS setting. A separate 0.25 GHz
four-port rerun checks the effect of that difference; archived data are never
silently replaced with corrected values.

The small `TutorialDrivenSim` subclass applies advanced linear-solver controls
whenever gsim writes a config, including during cloud packaging. This is needed
because this gsim revision gives MUMPS one correction step by default. Changing
only an already-written JSON file would lose these settings on regeneration.
The subclass contains no PDK layer or material names.


MUMPS is the sparse direct-factorization backend. Palace uses that factorization
inside **generalized minimal residual (GMRES)**, an iterative linear solver.
A preconditioner is an auxiliary, easier-to-solve matrix that helps GMRES;
its approximation must not be confused with the physical Maxwell operator.
Every accepted field solve must still meet the residual tolerance for the full
complex system. **Message Passing Interface (MPI)** coordinates worker
processes; OpenMP supplies CPU threads within each process.


In [ ]:
@dataclass(frozen=True)
class Settings:
    fmin: float = 0.25e9
    fmax: float = 200e9
    points: int = 800
    adaptive_tol: float = 1e-5
    refined_um: float = 6.0
    max_um: float = 100.0
    linear_solver: str = "MUMPS"
    linear_tolerance: float = 1e-8
    linear_max_iterations: int = 5
    real_preconditioner: bool = False
    order: int = 2
    air_um: float = 150.0


FULL_SETTINGS = Settings()
# Resource-bounded profile used for the recorded dense comparison.
BASE = replace(FULL_SETTINGS, order=1, refined_um=8.0, max_um=100.0)


from pydantic import Field


class TutorialDrivenSim(DrivenSim):
    """Keep advanced linear controls when local/cloud execution rewrites config."""

    tutorial_linear: dict = Field(default_factory=dict)

    def write_config(self, **kwargs):
        path = super().write_config(**kwargs)
        config = json.loads(path.read_text())
        config["Solver"]["Linear"].update(self.tutorial_linear)
        path.write_text(json.dumps(config, indent=2))
        return path


def configure_sim(
    component, stack, signal_key, return_key, settings, outdir, material_overrides=None
):
    if signal_key == return_key or not {signal_key, return_key} <= set(stack.layers):
        raise ValueError(
            "Select distinct signal and return layers present in the stack"
        )
    if stack.layers[signal_key].z_bounds[0] <= stack.layers[return_key].z_bounds[1]:
        raise ValueError("This interlayer example requires signal above return")
    sim = TutorialDrivenSim()
    sim.set_output_dir(str(outdir))
    sim.set_geometry(component)
    sim.set_stack(build_palace_layer_stack(stack))
    for name, properties in (material_overrides or {}).items():
        sim.set_material(name, **properties)
    sim.set_airbox(
        margin_x=settings.air_um,
        margin_y=settings.air_um,
        z_above=settings.air_um,
        z_below=settings.air_um,
    )
    for name in PORTS:
        sim.add_port(
            name,
            from_layer=return_key,
            to_layer=signal_key,
            geometry="interlayer",
            impedance=Z_SE,
            excited=True,
        )
    sim.set_driven(
        fmin=settings.fmin,
        fmax=settings.fmax,
        num_points=settings.points,
        scale="linear",
        adaptive_tol=settings.adaptive_tol,
        adaptive_max_samples=80,
        reference_impedance=Z_SE,
    )
    sim.solver.order = settings.order
    sim.solver.linear.solver_type = settings.linear_solver
    sim.solver.linear.tolerance = settings.linear_tolerance
    sim.tutorial_linear = {
        "MaxIts": settings.linear_max_iterations,
        "PCMatReal": settings.real_preconditioner,
        "ComplexCoarseSolve": not settings.real_preconditioner,
    }
    if settings.real_preconditioner:
        sim.tutorial_linear["MaxSize"] = settings.linear_max_iterations
    return sim


# A distinct synthetic PDK checks that geometry/stack/port setup uses supplied
# layer names and coordinates. This is a setup check, not another device example.
from gdsfactory.technology import LayerLevel
from gdsfactory.technology import LayerStack as GfLayerStack

synthetic = gf.Pdk(
    name="signal_integrity_synthetic",
    layer_stack=GfLayerStack(
        layers={
            "return_sheet": LayerLevel(
                layer=(91, 0), zmin=2, thickness=0.5, material="aluminum"
            ),
            "trace_sheet": LayerLevel(
                layer=(92, 0), zmin=9, thickness=1, material="aluminum"
            ),
        }
    ),
)
try:
    synthetic.activate()
    small = replace(GEO, length=200, gap_x=100, gap_length=20)
    dummy = make_interconnect(small, (92, 0), (91, 0), gap=True)
    dummy_stack = resolve_layer_stack(
        dummy,
        pdk=synthetic,
        use_pdk_overlay=False,
        dielectrics=[{"name": "bulk", "material": "sio2", "zmin": 0, "zmax": 11}],
    )
    dummy_sim = configure_sim(
        dummy,
        dummy_stack,
        "trace_sheet",
        "return_sheet",
        BASE,
        OUTPUT / "synthetic-setup",
    )
    assert dummy_sim.stack.layers["trace_sheet"].zmin == 9
    assert all(
        p.from_layer == "return_sheet" and p.to_layer == "trace_sheet"
        for p in dummy_sim.ports
    )
    assert all(p.excited for p in dummy_sim.ports)
finally:
    PDK.activate()
print(
    "Generic geometry and port setup accepted a synthetic PDK with different names/heights."
)

# Planning estimate only: actual delay is extracted from the solved baseline.
v_estimate = 1.5e8
frequency_step = (BASE.fmax - BASE.fmin) / (BASE.points - 1)
print(
    f"df={frequency_step / 1e9:g} GHz; periodic time span={1 / frequency_step * 1e12:g} ps"
)
print(
    f"Approximate sampling={1 / (2 * BASE.fmax) * 1e12:g} ps; working resolution={2 / BASE.fmax * 1e12:g} ps"
)
print(f"Estimated gap round trip={2 * GEO.gap_x * 1e-6 / v_estimate * 1e12:g} ps")

## 7. Run or reload complete experiments

`preview` leaves the expensive stages unexecuted. In `local`/`cloud` mode, matching
completed runs are reused only when their manifest and data hash match. A different
geometry, mesh setting, dependency version, or stack produces a different directory.
`load` requires a previously saved matching result. Partial or failed solves are
recorded with their status and are never treated as completed results.

The raw Palace files remain in the run directory. A compact NPZ (NumPy archive)
contains the complete network; a JSON manifest records its provenance. A Touchstone
`.s4p` file allows importing the network in other RF tools. No symmetry completion
is used. Reference impedances are read from results, not relabeled during export.

`recorded` loads the 8 µm first-order pair, checking data hashes, geometry, and
key settings. Original solver warnings remain visible in the executed-results
section. A completed data archive is not itself a validation certificate.

In [ ]:
def load_recorded_case(label):
    candidates = []
    for path in OUTPUT.glob(label + "-*"):
        manifest_file = path / "manifest.json"
        if manifest_file.exists():
            manifest = json.loads(manifest_file.read_text())
            if manifest.get("status") == "complete":
                candidates.append((path, manifest))
    if not candidates:
        return None, None, None
    if len(candidates) != 1:
        raise ValueError(f"Ambiguous recorded case {label}; select an explicit archive")
    directory, manifest = candidates[0]
    data_path = directory / "network.npz"
    if hashlib.sha256(data_path.read_bytes()).hexdigest() != manifest["sha256"]:
        raise ValueError(f"Data hash mismatch for {label}")
    with np.load(data_path, allow_pickle=False) as saved:
        net = rf.Network(f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz")
        net.port_names = saved["ports"].tolist()
    check_network(net)
    return net, manifest, directory


def json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, default=str)


def stack_record(stack):
    return {
        "pdk": stack.pdk_name,
        "overlay": stack.overlay,
        "notes": list(stack.notes),
        "layers": {k: asdict(v) for k, v in stack.layers.items()},
        "dielectrics": [asdict(d) for d in stack.dielectrics],
        "materials": dict(stack.materials),
    }


def validate_generated_config(config):
    ports = config["Boundaries"]["LumpedPort"]
    ids = [p.get("Excitation") for p in ports]
    if (
        len(ports) != 4
        or any(type(i) is not int for i in ids)
        or set(ids) != {1, 2, 3, 4}
    ):
        raise ValueError(f"Expected four independent excitations, got {ids}")
    if not all(np.isclose(p["R"], Z_SE) for p in ports):
        raise ValueError("Generated port impedances do not match the experiment")


def run_case(label, geometry, gap, settings, bulk=BULK):
    if EXECUTION == "recorded":
        recorded_label = {
            "intact": "refine8-intact",
            "disrupted": "refine8-disrupted",
        }.get(label)
        if recorded_label is None:
            return None  # The full proposed convergence suite was not completed.
        net, manifest, _ = load_recorded_case(recorded_label)
        if net is not None:
            if manifest["spec"]["geometry_um"] != asdict(geometry):
                raise ValueError("Recorded geometry differs from the tutorial controls")
            for key in (
                "fmin",
                "fmax",
                "points",
                "order",
                "refined_um",
                "max_um",
                "air_um",
            ):
                if manifest["spec"]["settings"][key] != getattr(settings, key):
                    raise ValueError(f"Recorded setting differs: {key}")
        return net
    component = make_interconnect(geometry, SIGNAL_LAYER, RETURN_LAYER, gap)
    stack = resolve_layer_stack(component, pdk=PDK, dielectrics=bulk)
    spec = {
        "schema": 1,
        "label": label,
        "geometry_um": asdict(geometry),
        "gap": gap,
        "settings": asdict(settings),
        "stack": stack_record(stack),
        "material_overrides": MATERIAL_OVERRIDES,
        "versions": VERSIONS,
        "gsim_revision": GSIM_REVISION,
        "ihp_source_revision": IHP_REVISION,
        "ihp_install": IHP_INSTALL,
        "palace_version": PALACE_VERSION,
        "ports": list(PORTS),
        "pairs": PAIRS,
        "z_se_ohm": Z_SE,
        "reference_planes_um": [0, geometry.length],
        "tdr": {"dc": "linear-real", "window": "hamming", "pad": 0},
    }
    signature = hashlib.sha256(json_text(spec).encode()).hexdigest()[:16]
    directory = OUTPUT / f"{label}-{signature}"
    manifest_path, data_path = directory / "manifest.json", directory / "network.npz"
    if EXECUTION == "preview":
        return None
    if manifest_path.exists() and data_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("status") == "complete":
            if json_text(manifest["spec"]) != json_text(spec):
                raise ValueError("Manifest mismatch")
            if hashlib.sha256(data_path.read_bytes()).hexdigest() != manifest["sha256"]:
                raise ValueError("Saved network hash mismatch")
            with np.load(data_path, allow_pickle=False) as saved:
                net = rf.Network(
                    f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz"
                )
                net.port_names = saved["ports"].tolist()
            check_network(net)
            return net
    if EXECUTION == "load":
        raise FileNotFoundError(f"No completed matching run in {directory}")
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {"status": "incomplete", "spec": spec}
    manifest_path.write_text(json_text(manifest))
    try:
        sim = configure_sim(
            component,
            stack,
            SIGNAL_KEY,
            RETURN_KEY,
            settings,
            directory,
            MATERIAL_OVERRIDES,
        )
        sim.mesh(
            preset="default",
            refined_mesh_size=settings.refined_um,
            max_mesh_size=settings.max_um,
            planar_conductors=False,
            fmax=settings.fmax,
            threads=1,
            surface_threads=1,
        )
        config_path = sim.write_config()
        config = json.loads(Path(config_path).read_text())
        validate_generated_config(config)
        run_logger = logging.getLogger("gsim.palace.base")
        log_handler = logging.FileHandler(directory / "solver.log")
        previous_level = run_logger.level
        run_logger.setLevel(logging.INFO)
        run_logger.addHandler(log_handler)
        try:
            if EXECUTION == "local":
                result = sim.run_local(
                    palace_executable=PALACE_EXECUTABLE,
                    num_processes=LOCAL_PROCESSES,
                    num_threads=2,
                )
            else:
                result = sim.run(check_cache=True)
        finally:
            run_logger.removeHandler(log_handler)
            log_handler.close()
            run_logger.setLevel(previous_level)
        from gsim.palace.validation import validate_solution

        evidence = validate_solution(
            (directory / "solver.log").read_text(),
            estimator_disabled=config["Solver"]["Linear"].get("EstimatorMaxIts") == 0,
            expected_excitations=[1, 2, 3, 4] if settings.adaptive_tol > 0 else [],
            exit_code=0 if EXECUTION == "local" else None,
        )
        manifest["solver_evidence"] = asdict(evidence)
        net = complete_network(result)
        target_f = np.linspace(settings.fmin, settings.fmax, settings.points)
        np.testing.assert_allclose(net.f, target_f, rtol=1e-8, atol=1)
        np.savez_compressed(
            data_path, f=net.f, s=net.s, z0=net.z0, ports=np.array(net.port_names)
        )
        net.write_touchstone(filename=str(directory / "network"))
        manifest.update(
            status="complete",
            sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(),
            config=str(config_path),
        )
        manifest_path.write_text(json_text(manifest))
    except Exception as exc:
        manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        manifest_path.write_text(json_text(manifest))
        raise
    else:
        return net


networks = {
    label: run_case(label, GEO, gap, BASE)
    for label, gap in [("intact", False), ("disrupted", True)]
}
EM_READY = all(net is not None for net in networks.values())
print(
    "Base EM runs complete."
    if EM_READY
    else "EM results INCOMPLETE: preview only; analytic checks are not EM validation."
)

## 8. Read the frequency-domain results

We plot $20\log_{10}|S|$ (dB). The display floor is −140 dB; it is not a numerical
accuracy claim. Each mixed-mode panel shows **all four entries in its 2×2 block**.

A reciprocal passive network with real power-wave references should have
$S\approx S^T$ and largest singular value $\sigma_{max}(S)\leq1$. Deviations are
useful diagnostics, not proof of physical correctness or a replacement for mesh
convergence. All 16 entries must be independently available before these checks.

In [ ]:
def db(value):
    return 20 * np.log10(np.maximum(np.abs(value), 1e-7))


if EM_READY:
    mixed = {label: mixed_mode(net)[1] for label, net in networks.items()}
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    for ax, (title, i, j) in zip(
        axes.flat,
        [
            ("Reflection S11", 0, 0),
            ("Transmission S31", 2, 0),
            ("NEXT S21", 1, 0),
            ("FEXT S41", 3, 0),
        ],
        strict=True,
    ):
        for label, net in networks.items():
            ax.plot(net.f / 1e9, db(net.s[:, i, j]), label=label)
        ax.set(title=title, xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
        ax.grid()
        ax.legend()
    fig.tight_layout()

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    for ax, (block, row, col) in zip(
        axes.flat, [("dd", 0, 0), ("dc", 0, 2), ("cd", 2, 0), ("cc", 2, 2)], strict=True
    ):
        for label, net in mixed.items():
            style = "-" if label == "intact" else "--"
            for k, (i, j) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
                ax.plot(
                    net.f / 1e9,
                    db(net.s[:, row + i, col + j]),
                    ls=style,
                    color=colors[k],
                    label=f"{label}: S{block}{i + 1}{j + 1}",
                )
        ax.set(title=f"S{block}", xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
        ax.grid()
        ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    show(
        pd.DataFrame(
            [
                {
                    "case": label,
                    "max |S-S^T|": np.max(np.abs(net.s - net.s.transpose(0, 2, 1))),
                    "max singular value": np.linalg.svd(net.s, compute_uv=False).max(),
                    "max mode conversion (linear)": max(
                        np.abs(mixed[label].s[:, :2, 2:]).max(),
                        np.abs(mixed[label].s[:, 2:, :2]).max(),
                    ),
                }
                for label, net in networks.items()
            ]
        )
    )
else:
    print("Frequency-domain EM plots pending completed runs.")

## 9. Locate the return-path interruption with TDR

We estimate the baseline's differential propagation delay from the phase slope:
$\tau_g=-d\arg(S_{dd21})/d\omega$. This includes the launches and may vary with
frequency. Its median is a **timing estimate**, not an independently known material
velocity. A uniform-path estimate of the gap echo is
$t_{gap}\approx2\tau_g x_{gap}/L$.

Overlay intact/disrupted reflected steps and apparent impedances, then examine
the difference in reflected steps. Estimate its first half-peak excursion within a fixed interior time gate that
excludes the launches and far-end reflection. Use the same gate for both slot
locations; the expected arrival is used only to assess the result, never to steer
the search. Multiple interior reflections can still make this estimator ambiguous. A moved-slot experiment below provides a stronger check.

We also vary DC extrapolation, window, and upper bandwidth. A feature that moves
substantially under those choices is not yet a reliably located discontinuity.

In [ ]:
def differential_delay(mm):
    transmission = mm.s[:, 1, 0]
    use = (
        (mm.f >= 0.1 * mm.f[-1])
        & (mm.f <= 0.6 * mm.f[-1])
        & (np.abs(transmission) > 0.05)
    )
    delays = -np.gradient(np.unwrap(np.angle(transmission)), 2 * np.pi * mm.f)
    if use.sum() < 5:
        raise ValueError("Insufficient transmitted signal to estimate delay")
    tau = float(np.median(delays[use]))
    if tau <= 0:
        raise ValueError("Nonpositive estimated delay: inspect ports, phase, and sweep")
    return tau


def gap_arrival(baseline, disrupted, one_way_delay, resolution_s, threshold):
    t, rb, _ = tdr(baseline.f, baseline.s[:, 0, 0], 2 * Z_SE)
    tg, rg, _ = tdr(disrupted.f, disrupted.s[:, 0, 0], 2 * Z_SE)
    np.testing.assert_allclose(t, tg)
    difference = np.abs(rg - rb)
    # The SAME gate applies at both slot locations. Do not center a search on
    # the expected slot time: that could make unchanged data appear to move.
    region = (t > 2 * resolution_s) & (t < 2 * one_way_delay - 2 * resolution_s)
    if not region.any() or difference[region].max() <= threshold:
        return np.nan
    level = max(threshold, 0.5 * difference[region].max())
    found = np.flatnonzero(region & (difference >= level))
    return float(t[found[0]]) if len(found) else np.nan


# Analytic timing probes exercise the same gate before using it on EM data.
# These are one-port reflected steps with known arrival times, not EM fixtures.
def timing_probe(arrival_s, amplitude=0.1):
    reflection = amplitude * np.exp(-2j * np.pi * F_ANALYTIC * arrival_s)
    return rf.Network(f=F_ANALYTIC, s=reflection[:, None, None], z0=100, f_unit="Hz")


zero_probe = timing_probe(0, amplitude=0)
first_probe, moved_probe = timing_probe(32e-12), timing_probe(48e-12)
first_time = gap_arrival(zero_probe, first_probe, 40e-12, 10e-12, 1e-4)
second_time = gap_arrival(zero_probe, moved_probe, 40e-12, 10e-12, 1e-4)
assert abs(first_time - 32e-12) < 10e-12
assert abs((second_time - first_time) - 16e-12) < 10e-12
assert np.isnan(gap_arrival(zero_probe, zero_probe, 40e-12, 10e-12, 1e-4))
print("Analytic reflection-location checks passed using a fixed time gate.")


if EM_READY:
    tau = differential_delay(mixed["intact"])
    predicted = 2 * tau * GEO.gap_x / GEO.length
    time_resolution = 2 / BASE.fmax
    print(f"Baseline median one-way group delay: {tau * 1e12:.2f} ps")
    print(
        f"Estimated slot round trip: {predicted * 1e12:.2f} ps (launch delay included)"
    )
    if min(predicted, 2 * tau - predicted) < 3 * time_resolution:
        print(
            "Resolution warning: extend the line or bandwidth before claiming localization."
        )
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, net in mixed.items():
        t, rho, z = tdr(net.f, net.s[:, 0, 0], 2 * Z_SE)
        axes[0].plot(t * 1e12, rho, label=label)
        axes[1].plot(t * 1e12, z, label=label)
    for ax in axes:
        ax.axvline(predicted * 1e12, color="k", ls=":", label="Estimated slot arrival")
        ax.set(
            xlim=(
                -2 * time_resolution * 1e12,
                min(3 * tau, 0.4 / frequency_step) * 1e12,
            ),
            xlabel="Time (ps)",
        )
        ax.grid()
        ax.legend()
    axes[0].set_ylabel("Reflected differential step / incident step")
    axes[1].set_ylabel("Apparent differential impedance (ohm)")
    fig.tight_layout()

    t, r_intact, _ = tdr(mixed["intact"].f, mixed["intact"].s[:, 0, 0], 2 * Z_SE)
    _, r_disrupted, _ = tdr(
        mixed["disrupted"].f, mixed["disrupted"].s[:, 0, 0], 2 * Z_SE
    )
    fig_delta, ax_delta = plt.subplots(figsize=(9, 3))
    ax_delta.plot(t * 1e12, r_disrupted - r_intact, label="Disrupted minus intact")
    ax_delta.axvline(
        predicted * 1e12, color="k", ls=":", label="Estimated slot arrival"
    )
    ax_delta.set(
        xlim=(-20, 3 * tau * 1e12),
        xlabel="Time (ps)",
        ylabel="Change in reflected step",
    )
    ax_delta.grid()
    ax_delta.legend()
    fig_delta.tight_layout()

    fig, ax = plt.subplots(figsize=(9, 4))
    net = mixed["disrupted"]
    for window, dc, fraction in [
        ("hamming", "linear", 1),
        ("hamming", "constant", 1),
        ("hann", "linear", 1),
        ("hamming", "linear", 0.5),
    ]:
        count = int(len(net.f) * fraction)
        t, rho, _ = tdr(
            net.f[:count], net.s[:count, 0, 0], 2 * Z_SE, window=window, dc=dc
        )
        ax.plot(
            t * 1e12,
            rho,
            label=f"{window}, DC={dc}, fmax={net.f[count - 1] / 1e9:g} GHz",
        )
    ax.set(
        xlim=(-20, 3 * tau * 1e12),
        xlabel="Time (ps)",
        ylabel="Reflected differential step",
    )
    ax.grid()
    ax.legend()
    fig.tight_layout()
else:
    print("EM TDR and DC/window/bandwidth sensitivity pending completed runs.")

## 10. Convergence: distinguish a resolved effect from a numerical artifact

Enable `RUN_CONVERGENCE` to run the matrix below. Each row runs both geometries.
The three meshes test spatial refinement; higher field order tests the launch;
the larger airbox tests the absorbing boundary; the deeper silicon region tests
substrate truncation; the tighter adaptive sweep tests frequency interpolation.
Direct solves at nine frequencies independently check that interpolation.

The finest common setup is the comparison reference, not ground truth. We use the
largest observed complex-S change as an **empirical numerical uncertainty indicator**
$\epsilon_S$. It is not a continuum error bound. A corresponding indicator for TDR
is the largest reflected-step change $\epsilon_\rho$ across these studies.

Acceptance criteria chosen **before** examining EM results:

* maximum study difference $\epsilon_S<0.01$ in absolute complex S;
* baseline mode conversion no greater than $3\epsilon_S$;
* gap-induced mixed-mode conversion change greater than $3\epsilon_S$;
* gap reflection detectable above $3\epsilon_\rho$;
* slot arrival and moved-slot delay shift agree with the propagation estimate
  within the working time resolution, with events separated from the launches.

The factors of three are pragmatic margins, not statistical confidence intervals.
An inconclusive/failed check calls for refinement or a better-resolved geometry,
not relaxed thresholds. The two gap locations must also be distinguishable in time.

In [ ]:
FINE = replace(BASE, refined_um=2.5, max_um=50)
STUDIES = {
    "coarse": replace(BASE, refined_um=10, max_um=150),
    "medium": BASE,
    "fine": FINE,
    "order3": replace(FINE, order=3),
    "airbox": replace(FINE, air_um=300),
    "substrate": FINE,
    "sweep": replace(FINE, adaptive_tol=1e-6),
    "direct": replace(FINE, points=9, adaptive_tol=0),
}
GAP_SHIFT = 1200.0
study_networks = {}
if RUN_CONVERGENCE:
    for study, settings in STUDIES.items():
        if study == "medium" and EM_READY:
            study_networks[study] = networks
            continue
        bulk = (
            [dict(d, zmin=-360.0) if d["name"] == "substrate" else d for d in BULK]
            if study == "substrate"
            else BULK
        )
        study_networks[study] = {
            label: run_case(f"{study}-{label}", GEO, gap, settings, bulk)
            for label, gap in [("intact", False), ("disrupted", True)]
        }
    moved = run_case(
        "moved-slot", replace(GEO, gap_x=GEO.gap_x + GAP_SHIFT), True, FINE
    )
else:
    moved = None


def interpolate_complex(net, frequencies):
    # Used only to compare direct anchors with the reported dense sweep;
    # this never fabricates a TDR spectrum from sparse anchors.
    return np.stack(
        [
            np.interp(frequencies, net.f, net.s[:, i, j].real)
            + 1j * np.interp(frequencies, net.f, net.s[:, i, j].imag)
            for i in range(4)
            for j in range(4)
        ],
        axis=-1,
    ).reshape(-1, 4, 4)


STUDY_READY = (
    bool(study_networks)
    and moved is not None
    and all(n is not None for pair in study_networks.values() for n in pair.values())
)
if STUDY_READY:
    reference = {label: mixed_mode(n)[1] for label, n in study_networks["fine"].items()}
    rows = []
    for study, pair in study_networks.items():
        for label, n in pair.items():
            mm = mixed_mode(n)[1]
            ref = reference[label]
            err_s = np.max(np.abs(mm.s - interpolate_complex(ref, mm.f)))
            err_rho = np.nan
            if study != "direct":
                t0, r0, _ = tdr(ref.f, ref.s[:, 0, 0], 2 * Z_SE)
                t1, r1, _ = tdr(mm.f, mm.s[:, 0, 0], 2 * Z_SE)
                np.testing.assert_allclose(t0, t1)
                err_rho = np.max(np.abs(r1 - r0))
            rows.append(
                {
                    "study": study,
                    "geometry": label,
                    "max complex S change": err_s,
                    "max reflected-step change": err_rho,
                }
            )
    convergence = pd.DataFrame(rows)
    show(convergence)
    eps_s = max(convergence["max complex S change"].max(), 1e-12)
    eps_rho = max(convergence["max reflected-step change"].max(), 1e-12)
    intact, disrupted = reference["intact"], reference["disrupted"]

    def conversion(s):
        return np.concatenate([s[:, :2, 2:].reshape(-1), s[:, 2:, :2].reshape(-1)])

    baseline_conversion = np.max(np.abs(conversion(intact.s)))
    conversion_change = np.max(np.abs(conversion(disrupted.s - intact.s)))
    tau = differential_delay(intact)
    predicted = 2 * tau * GEO.gap_x / GEO.length
    moved_predicted = 2 * tau * (GEO.gap_x + GAP_SHIFT) / GEO.length
    resolution_s = 2 / FINE.fmax
    arrival = gap_arrival(intact, disrupted, tau, resolution_s, 3 * eps_rho)
    moved_mm = mixed_mode(moved)[1]
    moved_arrival = gap_arrival(intact, moved_mm, tau, resolution_s, 3 * eps_rho)
    expected_shift = moved_predicted - predicted
    separation_ok = (
        min(predicted, 2 * tau - moved_predicted) > 3 * resolution_s
        and expected_shift > resolution_s
    )
    checks = {
        "absolute S convergence < 0.01": eps_s < 0.01,
        "symmetric conversion within empirical tolerance": baseline_conversion
        <= 3 * eps_s,
        "asymmetric conversion change resolved": conversion_change > 3 * eps_s,
        "gap event timing resolved": separation_ok
        and np.isfinite(arrival)
        and abs(arrival - predicted) <= resolution_s,
        "moved gap shift resolved": separation_ok
        and np.isfinite(moved_arrival)
        and abs((moved_arrival - arrival) - expected_shift) <= resolution_s,
    }
    show(pd.Series(checks, name="Passed").to_frame())
    print(f"Empirical eps_S={eps_s:.3g}; eps_rho={eps_rho:.3g}")
    print(
        f"Gap arrival={arrival * 1e12:.2f} ps; moved arrival={moved_arrival * 1e12:.2f} ps"
    )
    OUTPUT.mkdir(parents=True, exist_ok=True)
    convergence.to_csv(OUTPUT / "convergence.csv", index=False)
    (OUTPUT / "validation.json").write_text(
        json_text(
            {
                "checks": {k: bool(v) for k, v in checks.items()},
                "eps_s": float(eps_s),
                "eps_rho": float(eps_rho),
                "arrival_s": float(arrival),
                "moved_arrival_s": float(moved_arrival),
            }
        )
    )
    print(
        "All EM criteria passed."
        if all(checks.values())
        else "EM validation INCONCLUSIVE/FAILED: inspect refinement and time resolution."
    )
else:
    print(
        "EM convergence validation INCOMPLETE. Enable RUN_CONVERGENCE and complete the runs."
    )

## 11. What can we conclude?

The analytic cells check mixed-mode algebra, port reordering, uniform-line impedance
and delay recovery, and a known reflection time. These checks validate the
postprocessing on those examples; they do not validate the IHP electromagnetic model.

After the EM runs, use the reported convergence results to decide whether the gap's
mode conversion and reflection are resolved. A small symmetry residual is an
empirical diagnostic. Apparent TDR impedance includes launch effects, finite
bandwidth, loss, coupling, and repeated reflections. The substrate assumptions must
be replaced or justified before making claims about fabricated IHP devices.

Useful next experiments are changing trace separation, slot offset, and return
width one at a time. A centered slot is a useful symmetry control. Eye diagrams
and **bit error rate (BER)** require transmitter/receiver/noise assumptions and are
outside this tutorial's validation claims.

### Sources and adaptations

* [AWS Palace: crosstalk between coplanar waveguides (CPWs)](https://awslabs.github.io/palace/stable/examples/cpw/):
  four-terminal coupling, lumped/wave-port comparisons, mesh/order sensitivity, and
  checking adaptive sweeps against direct solves. Its sapphire CPW is not our IHP
  geometry. We do not assume its numerical curves transfer to this stack.
* [Palace v0.18.1 CPW configuration](https://github.com/awslabs/palace/blob/v0.18.1/examples/cpw/cpw_lumped_uniform.json):
  explicit excitation IDs. Our full mixed-mode matrix requires all four excitations.
* PalaceToolkit, `docs/examples/signal_integrity.ipynb`: inspiration for a common
  variant builder, separate mesh/solve/analysis stages, and consistent comparisons.
  Its bent-antenna geometry and passband OOK (on-off keying) eye/BER model are not
  used as a TDR algorithm. This notebook does not depend on PalaceToolkit.
* [scikit-rf mixed-mode conversion](https://scikit-rf.readthedocs.io/en/latest/api/generated/skrf.network.Network.se2gmm.html)
  and [step response](https://scikit-rf.readthedocs.io/en/latest/api/generated/skrf.network.Network.step_response.html).
* [scikit-rf TDR tutorial](https://scikit-rf.readthedocs.io/en/latest/examples/networktheory/Time%20domain%20reflectometry,%20measurement%20vs%20simulation.html):
  DC treatment, windowing, and time-resolution considerations.

## 12. Executed experiments: what the evidence supports

The local run used Palace **v0.18.0**. Two dense first-order mesh levels (12/150 µm
and 8/100 µm refined/background sizes) and nine direct frequency anchors were run
for the intact and disrupted return planes. The 6 µm second-order direct run was
stopped when host memory reached approximately 55 GiB and swapping began. The
initial iterative run was stopped after benchmarking a faster direct solver.
Neither stopped run supplies a completed network.

A second-order spot check on the **same 12 µm mesh** produced one complete
excitation column at 25 GHz before that study was stopped. It is adverse evidence
about spatial discretization, not a complete four-port validation. The remaining
second-order frequencies/excitations and the proposed airbox/substrate sensitivity
suite were not completed. Thus the full issue acceptance criteria are **not met**.

The tables below distinguish complete matrices, adaptive training convergence,
linear residual warnings, direct-sweep agreement, and mesh/order sensitivity. The
moved-slot experiment is an empirical timing check on the coarse model. Matching
its expected delay cannot establish continuum accuracy when discretization checks
fail.

The nine direct-check frequencies are not all on the 250 MHz output grid.
Both linear and cubic interpolation comparisons are reported: the former includes
a visible interpolation error of its own. These are checks at selected frequencies,
not rigorous error bounds over the full band; the endpoints are training points.

In [ ]:
from scipy.interpolate import CubicSpline

from gsim.palace.validation import validate_solution

RECORDED_LABELS = {
    "coarse intact": "pilot-mumps-intact",
    "coarse disrupted": "pilot-mumps-disrupted",
    "fine intact": "refine8-intact",
    "fine disrupted": "refine8-disrupted",
    "direct intact": "direct12-intact",
    "direct disrupted": "direct12-disrupted",
    "moved slot": "moved12",
}
recorded = {key: load_recorded_case(label) for key, label in RECORDED_LABELS.items()}
quality_rows = []
for name, (net, manifest, directory) in recorded.items():
    if net is None:
        quality_rows.append({"case": name, "data": "INCOMPLETE"})
        continue
    config = json.loads((directory / "config.json").read_text())
    log_file = directory / "solver.log"
    report = validate_solution(
        log_file.read_text() if log_file.exists() else "",
        estimator_disabled=config["Solver"]["Linear"].get("EstimatorMaxIts") == 0,
        expected_excitations=[1, 2, 3, 4]
        if manifest["spec"]["settings"]["adaptive_tol"] > 0
        else [],
        exit_code=0,
    )
    stats = json.loads((directory / "output/palace/palace.json").read_text())["Problem"]
    quality_rows.append(
        {
            "case": name,
            "data": "complete",
            "frequencies": len(net.f),
            "field order": config["Solver"]["Order"],
            "unknowns": stats["DegreesOfFreedom"],
            "linear checks": report.core.status,
            "adaptive checks": ", ".join(a.check.status for a in report.adaptive)
            or "direct",
            "adaptive samples": ", ".join(str(a.sample_count) for a in report.adaptive),
            "max reciprocity residual": np.abs(net.s - net.s.transpose(0, 2, 1)).max(),
            "max singular value": np.linalg.svd(net.s, compute_uv=False).max(),
        }
    )
quality = pd.DataFrame(quality_rows)
show(quality)
if OUTPUT.exists():
    quality.to_csv(OUTPUT / "recorded-run-quality.csv", index=False)

required = [
    "coarse intact",
    "coarse disrupted",
    "fine intact",
    "fine disrupted",
    "direct intact",
    "direct disrupted",
]
OBSERVED_READY = all(recorded[name][0] is not None for name in required)
if OBSERVED_READY:
    mm_recorded = {name: mixed_mode(recorded[name][0])[1] for name in required}
    comparisons = []
    for geometry in ("intact", "disrupted"):
        coarse = recorded["coarse " + geometry][0]
        fine = recorded["fine " + geometry][0]
        direct = recorded["direct " + geometry][0]
        np.testing.assert_allclose(coarse.f, fine.f)
        interp = interpolate_complex(coarse, direct.f)
        cubic = CubicSpline(coarse.f, coarse.s, axis=0)(direct.f)
        differences = np.abs(fine.s - coarse.s)
        comparisons.append(
            {
                "geometry": geometry,
                "max direct vs adaptive complex S": np.abs(interp - direct.s).max(),
                "max direct vs adaptive (cubic)": np.abs(cubic - direct.s).max(),
                "max mesh change, all frequencies": differences.max(),
                "max mixed-mode mesh change": np.abs(
                    mm_recorded["fine " + geometry].s
                    - mm_recorded["coarse " + geometry].s
                ).max(),
                "max mesh change, <=50 GHz": differences[fine.f <= 50e9].max(),
                "max mesh change, <=100 GHz": differences[fine.f <= 100e9].max(),
            }
        )
    show(pd.DataFrame(comparisons))
    pd.DataFrame(comparisons).to_csv(
        OUTPUT / "recorded-mesh-comparison.csv", index=False
    )
    rows = []
    for frequency in (25e9, 50e9, 100e9, 150e9, 200e9):
        for geometry in ("intact", "disrupted"):
            net = recorded["fine " + geometry][0]
            mm = mm_recorded["fine " + geometry]
            i = np.argmin(np.abs(net.f - frequency))
            rows.append(
                {
                    "geometry": geometry,
                    "GHz": net.f[i] / 1e9,
                    "Sdd21 dB": float(db(mm.s[i, 1, 0])),
                    "Sdd11 dB": float(db(mm.s[i, 0, 0])),
                    "NEXT dB": float(db(net.s[i, 1, 0])),
                    "FEXT dB": float(db(net.s[i, 3, 0])),
                    "max conversion amplitude": max(
                        np.abs(mm.s[i, :2, 2:]).max(), np.abs(mm.s[i, 2:, :2]).max()
                    ),
                }
            )
    show(pd.DataFrame(rows))
    pd.DataFrame(rows).to_csv(OUTPUT / "recorded-frequency-metrics.csv", index=False)
    numerical_change = max(row["max mixed-mode mesh change"] for row in comparisons)
    fine_b, fine_g = mm_recorded["fine intact"], mm_recorded["fine disrupted"]
    delta = fine_g.s - fine_b.s
    gap_conversion = max(np.abs(delta[:, :2, 2:]).max(), np.abs(delta[:, 2:, :2]).max())
    print(
        f"Largest mesh change: {numerical_change:.6g}; gap-induced conversion change: {gap_conversion:.6g}"
    )
    print("Convergence criterion |delta S| < 0.01:", numerical_change < 0.01)
    print(
        "Gap conversion exceeds 3 x mesh-change indicator:",
        gap_conversion > 3 * numerical_change,
    )
    # Compare TDR differences across mesh levels using the same frequency grid/window.
    fig, ax = plt.subplots(figsize=(10, 4))
    curves = {}
    for level in ("coarse", "fine"):
        b, g = mm_recorded[level + " intact"], mm_recorded[level + " disrupted"]
        t, rb, _ = tdr(b.f, b.s[:, 0, 0], 2 * Z_SE)
        _, rg, _ = tdr(g.f, g.s[:, 0, 0], 2 * Z_SE)
        curves[level] = (rb, rg)
        ax.plot(t * 1e12, rg - rb, label=level + ": disrupted minus intact")
    tau_fine = differential_delay(fine_b)
    predicted = 2 * tau_fine * GEO.gap_x / GEO.length
    ax.axvline(predicted * 1e12, color="k", ls=":", label="Estimated slot round trip")
    ax.set(xlim=(-10, 120), xlabel="Time (ps)", ylabel="Change in reflected step")
    ax.grid()
    ax.legend()
    fig.tight_layout()
    rho_mesh = max(
        np.abs(curves["fine"][i] - curves["coarse"][i]).max() for i in (0, 1)
    )
    print(
        f"One-way delay: {tau_fine * 1e12:.3f} ps; predicted slot echo: {predicted * 1e12:.3f} ps"
    )
    print(f"Maximum TDR step change across meshes: {rho_mesh:.6g}")
    moved_net = recorded["moved slot"][0]
    if moved_net is not None:
        cb, cg = mm_recorded["coarse intact"], mm_recorded["coarse disrupted"]
        cm = mixed_mode(moved_net)[1]
        tau_coarse = differential_delay(cb)
        # First report the raw timing estimator, then apply the mesh uncertainty gate.
        a0 = gap_arrival(cb, cg, tau_coarse, 2 / BASE.fmax, 1e-4)
        a1 = gap_arrival(cb, cm, tau_coarse, 2 / BASE.fmax, 1e-4)
        expected_shift = 2 * tau_coarse * 1200 / GEO.length
        robust = gap_arrival(cb, cg, tau_coarse, 2 / BASE.fmax, 3 * rho_mesh)
        print(f"Raw gap/moved-gap arrivals: {a0 * 1e12:.3f}, {a1 * 1e12:.3f} ps")
        print(
            f"Observed shift: {(a1 - a0) * 1e12:.3f} ps; expected: {expected_shift * 1e12:.3f} ps"
        )
        print("Gap detected above 3 x mesh TDR variation:", bool(np.isfinite(robust)))
        fig, ax = plt.subplots(figsize=(10, 4))
        t, rb, _ = tdr(cb.f, cb.s[:, 0, 0], 2 * Z_SE)
        for title, net in [("slot at 2400 um", cg), ("slot at 3600 um", cm)]:
            _, r, _ = tdr(net.f, net.s[:, 0, 0], 2 * Z_SE)
            ax.plot(t * 1e12, r - rb, label=title)
        ax.set(
            xlim=(-10, 120),
            xlabel="Time (ps)",
            ylabel="Change from intact reflected step",
        )
        ax.grid()
        ax.legend()
        fig.tight_layout()
else:
    print("Recorded comparison INCOMPLETE: missing one or more full matrices.")

if OBSERVED_READY:
    (OUTPUT / "recorded-analysis.json").write_text(
        json_text(
            {
                "status": "spatial_convergence_failed"
                if numerical_change >= 0.01
                else "full_validation_incomplete",
                "full_convergence_suite_complete": False,
                "max_mixed_mode_mesh_change": float(numerical_change),
                "s_convergence_threshold": 0.01,
                "gap_conversion_change": float(gap_conversion),
                "max_tdr_mesh_change": float(rho_mesh),
                "fine_one_way_delay_s": float(tau_fine),
                "coarse_raw_slot_shift_s": float(a1 - a0)
                if moved_net is not None
                else None,
                "coarse_expected_slot_shift_s": float(expected_shift)
                if moved_net is not None
                else None,
                "timing_feature_above_three_times_mesh_indicator": bool(
                    np.isfinite(robust)
                )
                if moved_net is not None
                else None,
                "note": "Mesh differences are empirical indicators, not error bounds. Second-order check is one excitation only.",
            }
        )
    )
    print("\nINTERPRETATION: Completed EM data; spatial convergence FAILED.")
    print("Adaptive/direct agreement and reciprocity do not establish mesh accuracy.")
    print("Treat crosstalk, conversion, and apparent impedance as exploratory values.")

### Separate low-frequency linear-algebra and higher-order checks

The original one-correction MUMPS solves sometimes missed the requested $10^{-8}$
relative residual tolerance at 0.25 GHz. Repeating that four-port point with up to
five corrections passed and changed the S-matrix by only about $10^{-10}$. This
explains why the linear warning has negligible impact here; the original logs
remain flagged rather than being rewritten as successful.

The partial second-order run below contributes only its independently solved
**first excitation column at 25 GHz**. All other entries in that unfinished run
are excluded. The comparison demonstrates a failed discretization check; it does
not certify the second-order answer as ground truth.

In [ ]:
correction_dir = OUTPUT / "low-frequency-correction-intact"
if (correction_dir / "network.npz").exists() and recorded["coarse intact"][
    0
] is not None:
    with np.load(correction_dir / "network.npz", allow_pickle=False) as saved:
        correction = saved["s"]
    original = recorded["coarse intact"][0].s[:1]
    report = validate_solution(
        (correction_dir / "solver.log").read_text(),
        estimator_disabled=True,
        exit_code=0,
    )
    print("Corrected low-frequency linear check:", report.core.status)
    print("max |S_corrected - S_original|:", np.abs(correction - original).max())

spot_dirs = list(OUTPUT.glob("order2-anchors-intact-*/output/palace/port-S.csv"))
if len(spot_dirs) == 1 and recorded["coarse intact"][0] is not None:
    frame = pd.read_csv(spot_dirs[0])
    frame.columns = [column.strip() for column in frame.columns]
    row = frame.loc[np.isclose(frame.iloc[:, 0], 25.0)].iloc[0]
    coarse = recorded["coarse intact"][0]
    i = np.argmin(np.abs(coarse.f - 25e9))
    spot_rows = []
    # Native solver order p1,p3,p2,p4; explicitly map responses to tutorial names.
    for native, name in enumerate(("p1", "p3", "p2", "p4"), start=1):
        magnitude = float(row[f"|S[{native}][1]| (dB)"])
        phase = float(row[f"arg(S[{native}][1]) (deg.)"])
        value = 10 ** (magnitude / 20) * np.exp(1j * np.deg2rad(phase))
        old = coarse.s[i, coarse.port_names.index(name), 0]
        spot_rows.append(
            {
                "response to p1": name,
                "GHz": 25,
                "first-order dB": float(db(old)),
                "second-order dB": magnitude,
                "complex change": abs(value - old),
            }
        )
    show(pd.DataFrame(spot_rows))
    print(
        "Partial check only: remaining second-order excitation columns were NOT completed."
    )

### Reproducing the recorded profiles

The original full-resolution profile remains in `FULL_SETTINGS`; it is not the
resource-bounded profile analyzed above. To launch fresh resource-bounded cases,
set `EXECUTION = "local"` (or `"cloud"`), select the runtime, and call the existing
`run_case` helper with these settings. Future runs use five correction iterations
and record their solver checks; their manifests will differ from the original
one-correction archives.

```python
coarse = replace(FULL_SETTINGS, order=1, refined_um=12, max_um=150)
finer = replace(FULL_SETTINGS, order=1, refined_um=8, max_um=100)
anchors = replace(coarse, points=9, adaptive_tol=0)
for profile, settings in [("coarse", coarse), ("finer", finer), ("anchors", anchors)]:
    for label, gap in [("intact", False), ("disrupted", True)]:
        run_case(f"repeat-{profile}-{label}", GEO, gap, settings)
run_case("repeat-moved", replace(GEO, gap_x=GEO.gap_x + 1200), True, coarse)
```

Keep all emitted manifests, meshes, port information, configurations, and logs.
A fresh run with a changed parameter is a new experiment, not a replacement for
an unfavorable convergence result.


## 13. Interpreting this run

**The simulations produced networks, but the discretization checks failed.**
Agreement between direct and adaptive frequency sweeps tests the frequency approximation
on a fixed mesh. Reciprocity and passivity are useful consistency diagnostics. None of
these tests establishes accuracy of the spatial finite-element approximation.

The intact-line mesh change alone reaches approximately **0.25 in complex single-ended
S**, versus the proposed 0.01 criterion. The symmetric baseline's largest cross-mode
amplitude falls from **0.094 to 0.037** on refinement, indicating substantial numerical
symmetry breaking. On the same coarse mesh, increasing the element order changes the
25 GHz far-end crosstalk from approximately **−42.3 to −23.3 dB**. Consequently, the slot's
crosstalk and conversion effects cannot yet be separated reliably from discretization error.

The coarse timing experiment is qualitatively consistent: moving the slot by 1.2 mm
moves the fixed-gate half-peak arrival from **30.0 to 47.5 ps**, a 17.5 ps shift compared
with the 16.1 ps estimate from group delay. The approximate Hamming-window resolution
is 10 ps. This is an exploratory timing observation, not a demonstrated localization
accuracy. The apparent impedance curve is likewise not a converged characteristic
impedance measurement.

### Numerical values from the finer first-order pair (exploratory)

At 100 GHz, differential transmission $S_{dd21}$ is **−4.204 dB intact** and
**−4.228 dB disrupted**. Differential reflection $S_{dd11}$ is **−17.673 dB**
and **−19.758 dB**, respectively. The slot therefore does not simply worsen
reflection at every frequency: its extra reflected wave interferes with the
existing reflections. Single-ended near-end/far-end crosstalk is
**−27.02/−16.01 dB intact**, versus **−25.55/−15.79 dB disrupted**.

The finer slot-induced reflected-step peak is **0.01428** at **32.48 ps**
(about 1.43% of the incident step); the apparent-impedance difference there
is **2.29 Ω**. The coarse peak is approximately 0.01702. The maximum change
in either absolute reflected-step curve across meshes is **0.05378**.
This conservative mesh-variation indicator is larger than the slot feature;
it is not a statistical confidence interval or a rigorous bound on the difference.

The largest mixed-mode mesh change is **0.3765**, while the slot-induced
cross-mode change is **0.05176**. Both the absolute 0.01 convergence criterion
and the requirement that the effect exceed three times the mesh indicator fail.
The saved tables retain these unfavorable checks alongside the timing plots.

### What to improve before using these numbers for design

1. Establish element-order and mesh convergence at direct frequency anchors first.
   The partial second-order result is a diagnostic, not a reference solution.
2. Use sufficient memory for the full higher-order problem, or establish a scalable
   iterative-solver configuration. The attempted full second-order direct solve
   approached 55 GiB host memory and was stopped when swapping began.
3. Repeat the dense intact/disrupted pair only after the direct checks settle;
   then complete the airbox and substrate-depth sensitivity studies.
4. Repeat the moved-slot and DC/window/bandwidth checks with the converged spectra.
   Require the slot-induced change to exceed the measured numerical variation.

All failed checks and stopped runs are retained. These results do **not** establish
that the physical slot has no effect; they establish that this discretization is
insufficient to quantify that effect.


## 14. Resolving spatial convergence

The first-order dense spectra above are retained as an unsuccessful convergence
study. Before repeating them, compare **complete four-port direct solves at
25, 112.5, and 200 GHz**. These frequencies are exactly present in the original
dense grid, so comparison does not require interpolation.

The recovery study compares second-order elements on coarser meshes with
first-order elements on finer meshes. Increasing field order and refining the
mesh are independent controls. A coarser second-order mesh may represent smooth
fields more efficiently, while the conductor edges still require geometric
resolution. Each row reports the actual solved unknown count and residual checks.
A completed run is not automatically a converged discretization.

The original issue requires all of the following:

| Requirement from #274 | Evidence required |
|---|---|
| IHP device example; PDK-independent simulation API | Pinned IHP setup plus generic geometry/stack/port functions and the synthetic PDK setup check |
| Configurable return-path gap; intact/disrupted comparison | Both geometries solved with the same materials, ports, frequency grid, and numerical profile |
| Sdd/Sdc/Scd/Scc, NEXT/FEXT, and TDR | Full matrices; explicitly named port pairs, impedances, and reference planes; plots of each quantity |
| TDR bandwidth, window, and DC policy | Uniform frequency grid, stated finite bandwidth, window/DC sensitivity, and analytic reconstruction checks |
| Mixed-mode conversion and inverse, including reordered ports | Independent matrix formula, scikit-rf conversion, inverse round trip, and permutations |
| Negligible symmetric conversion within mesh tolerance | Converging baseline, symmetry diagnostic, and changes in the conversion blocks across independently refined discretizations |
| Resolved asymmetric-gap change | Gap-induced complex conversion changes compared with numerical changes for the same observable |
| Analytic impedance/delay and gap location | Analytic checks, resolved EM echo, and a measured location shift consistent with round-trip propagation |

We retain an absolute **0.01 complex-S convergence target** for the network.
The numerical variation in the **conversion blocks** and reflected step is also
reported separately, because an error in through-wave phase and an error in mode
conversion are different observables. These variations are empirical convergence
indicators, not rigorous error bounds. Passing only a symmetry diagnostic cannot
establish accuracy of the entire network.


A real-only MUMPS preconditioner is an optional memory reduction strategy:
`real_preconditioner=True` factors the real part while GMRES still solves the
full complex Maxwell system. It requires more Krylov iterations. The 25 GHz
probe on the original second-order mesh agreed with the full complex
factorization to $2.7\times10^{-8}$ in the solved S-parameter column. Use a
strict complex residual tolerance and verify the resulting networks; changing a
preconditioner does not improve the spatial approximation by itself.


The completed 24 µm second-order anchor run passes its linear checks, but its
largest symmetric cross-mode amplitude at the anchors is **0.02951**. Thus this
coarser second-order discretization is not yet a sufficient reference. Finer
second-order and first-order comparisons, plus higher-order cross-checks, are
tracked below. Set `RUN_RECOVERY=True`, select `RECOVERY_TO_RUN`, and use a new
`RECOVERY_PREFIX` with local/cloud execution to reproduce these experiments
without relying on external helper scripts. The listed profiles use the same
geometry and material assumptions as the original study.


Long runs can be interrupted after some independent frequency/excitation solves
have converged. Such columns are preserved, but a network is marked complete
only after all 16 responses exist at every requested frequency. Recovery logs
identify the retained columns and the additional solves. The finer 4 µm
first-order profile uses the tested real preconditioner with a strict complex
GMRES residual tolerance to limit factorization memory.


### Why a small transverse error matters along a long line

For a roughly quasi-TEM line, the per-length inductance $L'$ and capacitance $C'$
give $\tau\approx\ell\sqrt{L'C'}$ and $Z_c\approx\sqrt{L'/C'}$. A small delay
error changes a through wave approximately by

$$|\Delta S|\approx |S|\,\omega |\Delta\tau|.$$

At 200 GHz and a 40 ps transit, a 0.1% delay error produces about 0.05 radians
of phase error, or $|\Delta S|\approx0.025$ for a through amplitude of 0.5.
That exceeds our 0.01 target even though the timing error is far below the
10 ps TDR resolution. A stable-looking TDR arrival therefore cannot establish
frequency-domain convergence. Resolving the fields near thin layers and conductor
corners matters as well as resolving the wavelength along the line. These
quasi-TEM estimates motivate the study; they are not error bounds for the lossy
three-dimensional Maxwell solve.

### Completed recovery results so far

The **4 µm first-order** network has 918,365 unknowns and passes its linear
checks, but the change from the 8 µm result reaches **0.14366** in mixed-mode S
at the direct anchors. Its baseline cross-mode amplitude reaches **0.02002**.
The **24 µm third-order** network has also been completed by recovering every
missing independent solve. Its baseline cross-mode amplitude is **0.01499**,
compared with **0.02951** for second order at the same nominal mesh sizes.
The second-to-third-order mixed-mode S change is **0.14277**. These results
still fail the network convergence target; neither is designated ground truth.


The completed **12 µm second-order** reference also passes its linear checks.
Its baseline cross-mode amplitude reaches **0.01620**, and its difference from
the 24 µm third-order network reaches **0.07240** in mixed-mode S at the anchors.
Thus the independent finer-mesh/higher-order comparison also fails the 0.01 target.
The recovery study has not yet established a usable reference discretization.


The completed **18 µm second-order** network has 1,022,508 unknowns and a
baseline cross-mode amplitude of **0.01988** at the anchors. The mixed-mode
change from 24 to 18 µm is **0.07502**, and from 18 to 12 µm is **0.11663**.
Every retained column passes its linear checks; this refinement sequence also
fails the spatial target. The next mesh experiments align prisms with the long
traces while refining conductor corners independently. Partial pilot columns
are exploratory until complete networks and independent refinements pass.


In [ ]:
# Fresh anchor experiments are sequential and can be memory intensive.
RUN_RECOVERY = False
RECOVERY_PREFIX = ""  # Use "repeat-" to keep fresh experiments separate.
ANCHOR_SETTINGS = replace(FULL_SETTINGS, fmin=25e9, points=3, adaptive_tol=0)
REAL_PC_SETTINGS = replace(
    ANCHOR_SETTINGS,
    real_preconditioner=True,
    linear_max_iterations=200,
    linear_tolerance=1e-9,
)
RECOVERY_SETTINGS = {
    "convergence-p2h24-intact": replace(
        ANCHOR_SETTINGS, order=2, refined_um=24, max_um=180
    ),
    "convergence-p2h18-intact": replace(
        ANCHOR_SETTINGS, order=2, refined_um=18, max_um=150
    ),
    "convergence-p1h4-real-intact": replace(
        REAL_PC_SETTINGS, order=1, refined_um=4, max_um=70
    ),
    "convergence-p2h12-real-intact": replace(
        REAL_PC_SETTINGS, order=2, refined_um=12, max_um=150
    ),
    "convergence-p3h24-real-intact": replace(
        REAL_PC_SETTINGS, order=3, refined_um=24, max_um=180
    ),
    "convergence-p3h18-real-intact": replace(
        REAL_PC_SETTINGS, order=3, refined_um=18, max_um=150
    ),
}
RECOVERY_TO_RUN = [
    "convergence-p2h24-intact",
    "convergence-p2h18-intact",
    "convergence-p1h4-real-intact",
]
if RUN_RECOVERY:
    if EXECUTION not in {"local", "cloud"}:
        raise ValueError("Set EXECUTION to local/cloud before launching recovery runs")
    for label in RECOVERY_TO_RUN:
        run_case(RECOVERY_PREFIX + label, GEO, False, RECOVERY_SETTINGS[label])

RECOVERY_PROFILES = {
    "first order, 12 um": "pilot-mumps-intact",
    "first order, 8 um": "refine8-intact",
    "first order, 4 um": "convergence-p1h4-real-intact",
    "second order, 24 um": "convergence-p2h24-intact",
    "second order, 18 um": "convergence-p2h18-intact",
    "second order, 12 um (real PC)": "convergence-p2h12-real-intact",
    "third order, 24 um (real PC)": "convergence-p3h24-real-intact",
    "third order, 18 um (real PC)": "convergence-p3h18-real-intact",
}
recovery_runs = {
    name: load_recorded_case(
        RECOVERY_PREFIX + label if label.startswith("convergence-") else label
    )
    for name, label in RECOVERY_PROFILES.items()
}


def recovery_physics_signature(manifest, directory):
    """Compare the generated physical problem while excluding numerical settings."""
    spec = manifest["spec"]
    config = json.loads((directory / "config.json").read_text())
    payload = {
        "geometry": spec["geometry_um"],
        "gap": spec["gap"],
        "air_um": spec["settings"]["air_um"],
        "dielectrics": spec["stack"]["dielectrics"],
        "layer_bounds": {
            key: [layer["zmin"], layer["zmax"]]
            for key, layer in spec["stack"]["layers"].items()
        },
        "length_scale": config["Model"]["L0"],
        "materials": config["Domains"]["Materials"],
        "conductivity": config["Boundaries"]["Conductivity"],
        "ports": config["Boundaries"]["LumpedPort"],
        "absorbing": config["Boundaries"]["Absorbing"],
        "port_names": spec["ports"],
        "pairs": spec["pairs"],
        "reference_planes": spec["reference_planes_um"],
    }
    return hashlib.sha256(json_text(payload).encode()).hexdigest()


recovery_reference_physics = None
recovery_rows = []
anchor_frequencies = np.array([25e9, 112.5e9, 200e9])
recovery_at_anchors = {}
for name, (net, manifest, directory) in recovery_runs.items():
    if net is None:
        recovery_rows.append({"profile": name, "status": "incomplete"})
        continue
    physics = recovery_physics_signature(manifest, directory)
    if recovery_reference_physics is None:
        recovery_reference_physics = physics
    if physics != recovery_reference_physics:
        raise ValueError(
            f"Physical model differs for {name}; compare numerical profiles of the same device"
        )
    indices = np.array(
        [np.argmin(np.abs(net.f - frequency)) for frequency in anchor_frequencies]
    )
    np.testing.assert_allclose(net.f[indices], anchor_frequencies, rtol=0, atol=1)
    mm = mixed_mode(net)[1]
    recovery_at_anchors[name] = mm.s[indices]
    report = validate_solution(
        (directory / "solver.log").read_text(), estimator_disabled=True, exit_code=0
    )
    stats = json.loads((directory / "output/palace/palace.json").read_text())["Problem"]
    recovery_rows.append(
        {
            "profile": name,
            "status": "complete",
            "unknowns": stats["DegreesOfFreedom"],
            "linear checks": report.core.status,
            "max cross-mode amplitude at anchors": max(
                np.abs(mm.s[indices, :2, 2:]).max(), np.abs(mm.s[indices, 2:, :2]).max()
            ),
        }
    )
show(pd.DataFrame(recovery_rows))
recovery_comparisons = []
for first, second in [
    ("first order, 12 um", "first order, 8 um"),
    ("first order, 8 um", "first order, 4 um"),
    ("second order, 24 um", "second order, 18 um"),
    ("second order, 24 um", "third order, 24 um (real PC)"),
    ("first order, 4 um", "third order, 24 um (real PC)"),
    ("first order, 4 um", "second order, 18 um"),
    ("second order, 18 um", "second order, 12 um (real PC)"),
    ("second order, 12 um (real PC)", "third order, 24 um (real PC)"),
    ("third order, 24 um (real PC)", "third order, 18 um (real PC)"),
]:
    if first not in recovery_at_anchors or second not in recovery_at_anchors:
        continue
    delta = recovery_at_anchors[first] - recovery_at_anchors[second]
    recovery_comparisons.append(
        {
            "profiles compared": first + " / " + second,
            "max complex mixed-mode S change": np.abs(delta).max(),
            "max conversion-block change": max(
                np.abs(delta[:, :2, 2:]).max(), np.abs(delta[:, 2:, :2]).max()
            ),
            "network target <0.01": bool(np.abs(delta).max() < 0.01),
        }
    )
if recovery_comparisons:
    show(pd.DataFrame(recovery_comparisons))
    pd.DataFrame(recovery_rows).to_csv(OUTPUT / "recovery-profiles.csv", index=False)
    pd.DataFrame(recovery_comparisons).to_csv(
        OUTPUT / "recovery-comparisons.csv", index=False
    )
else:
    print("Independent recovery comparisons pending complete four-port solves.")

## 15. Check the EM solver against an exactly solvable line

The earlier analytic tests exercise postprocessing. Here we also solve Maxwell's
equations with Palace for an ideal **transverse electromagnetic (TEM)** line.
This is a verification fixture; the IHP interconnect remains the device example.

Take a homogeneous box of length $\ell$, width $W$, height $H$, and relative
permittivity $\epsilon_r$. The top and bottom plates are **perfect electric
conductors (PEC)**, so their tangential electric field vanishes. The sidewalls
have the natural **perfect magnetic conductor (PMC)** condition, so their
tangential magnetic field vanishes. End ports span the entire cross-section.
The uniform fields $E_z,H_y$ then give an exact TEM solution, without fringing:

$$Z_c=\sqrt{\frac{\mu_0}{\epsilon_0\epsilon_r}}\frac{H}{W},\qquad
\tau=\frac{\ell\sqrt{\epsilon_r}}{c},\qquad
\begin{pmatrix}A&B\\C&D\end{pmatrix}=
\begin{pmatrix}\cosh(j\omega\tau)&Z_c\sinh(j\omega\tau)\\
\sinh(j\omega\tau)/Z_c&\cosh(j\omega\tau)\end{pmatrix}.$$

Section 3 converts this ABCD matrix to an exact two-port scattering matrix.
We compare **all four complex S entries** from two independent excitations.
The short matched fixture uses $\ell=1$ mm, $W=20$ µm, $H=10$ µm,
$\epsilon_r=4.1$, and port resistance $Z_c\approx93.03\,\Omega$.
Refine the axial spacing $h_x$ and field order using tetrahedra, triangular
prisms, and hexahedra. Geometry order stays one. Different element types have
different degrees of freedom; equal $h_x$ is not an equal-cost comparison.

**Recorded result:** at 25, 112.5, and 200 GHz the maximum complex-S error on
second-order prism/hex meshes decreases from $1.89\times10^{-4}$ at
$h_x=50$ µm to $1.19\times10^{-5}$ at 25 µm. First-order prism/hex errors are
0.0624, 0.0158, and 0.00398 at 50, 25, and 12.5 µm. Tetrahedral errors also
decrease (table below). Prism/hex agreement checks element handling for this
uniform field; it says nothing about corner resolution in the IHP device.

For an EM-to-TDR check, lengthen the same box to 6 mm and set both reference
resistances to $50\,\Omega$. The impedance mismatch produces a reflected step
with a plateau at $Z_c$ before the far-end round trip $2\tau$. Use the same
0.25–200 GHz grid, Hamming window, and DC reconstruction as the device analysis.
Recover the plateau impedance within 2% and the first transmitted impulse within
the stated 10 ps time resolution. This also tests the phase/units of simulated
S-parameters. These checks do not establish IHP mesh convergence.

**Reproduction:** `RUN_TEM_CALIBRATION=True` runs these small local CPU problems.
All mesh, configuration, execution, and analysis code is below. It needs Palace
at `PALACE_EXECUTABLE`; it does not read another notebook or a template config.
The lossless symmetric fixture can trigger an exactly dependent shared reduced
basis in Palace v0.18.0. The dense check therefore builds a separate reduced-order
model (ROM) per excitation and assembles the independently solved columns.
No missing column is inferred by reciprocity. A single-excitation log has an
unnumbered adaptive summary in this runtime; validate that sole summary against
the explicit source index in its saved config.


**Recorded dense EM/TDR result:** both independent columns passed the linear and
adaptive checks. The full 800-frequency matrix differs from the exact line by
at most **8.08 × 10⁻⁵** in complex S. TDR recovers **92.37 Ω** against
**93.03 Ω** (0.70% error), and **39.98 ps** against **40.52 ps** (0.55 ps
error, within the 10 ps working resolution). The impedance offset includes the
finite bandwidth, window, and DC treatment. These are successful ideal-line
verification results; the IHP gap response is still under convergence study.


In [ ]:
import os

from scipy.constants import c as light_speed
from scipy.constants import epsilon_0, mu_0

RUN_TEM_CALIBRATION = False
TEM_ROOT = OUTPUT / "tem-calibration"
TEM_EPS = 4.1
TEM_ZC = np.sqrt(mu_0 / (epsilon_0 * TEM_EPS)) * 0.5


def tem_write_box(path, kind, hx, L=1000.0, W=20.0, H=10.0):
    if kind not in {"tet", "prism", "hex"} or min(hx, L, W, H) <= 0:
        raise ValueError("Choose a supported element and positive dimensions")
    xs = np.linspace(0, L, int(np.ceil(L / hx)) + 1)
    ys = np.linspace(-W / 2, W / 2, 3)
    zs = np.linspace(0, H, 3)
    nx = len(xs) - 1
    grid = np.arange(1, (nx + 1) * 9 + 1).reshape(nx + 1, 3, 3)
    xyz = np.stack(np.meshgrid(xs, ys, zs, indexing="ij"), axis=-1).reshape(-1, 3)
    cells = []
    for i in range(nx):
        for j in range(2):
            for k in range(2):
                n = [
                    grid[i, j, k],
                    grid[i + 1, j, k],
                    grid[i + 1, j + 1, k],
                    grid[i, j + 1, k],
                    grid[i, j, k + 1],
                    grid[i + 1, j, k + 1],
                    grid[i + 1, j + 1, k + 1],
                    grid[i, j + 1, k + 1],
                ]
                if kind == "hex":
                    cells.append((5, n))
                elif kind == "tet":
                    for ids in [
                        (0, 1, 2, 6),
                        (0, 2, 3, 6),
                        (0, 3, 7, 6),
                        (0, 7, 4, 6),
                        (0, 4, 5, 6),
                        (0, 5, 1, 6),
                    ]:
                        cells.append((4, [n[q] for q in ids]))  # noqa: PERF401 -- explicit element connectivity
                else:
                    # Split the y-z cross-section along its positive diagonal; sweep in x.
                    a = [
                        grid[i, j, k],
                        grid[i, j + 1, k],
                        grid[i, j + 1, k + 1],
                        grid[i, j, k + 1],
                    ]
                    b = [
                        grid[i + 1, j, k],
                        grid[i + 1, j + 1, k],
                        grid[i + 1, j + 1, k + 1],
                        grid[i + 1, j, k + 1],
                    ]
                    for ids in [(0, 1, 2), (0, 2, 3)]:
                        cells.append((6, [a[q] for q in ids] + [b[q] for q in ids]))  # noqa: PERF401 -- explicit connectivity
    faces = {}
    templates = {
        5: [
            (0, 3, 2, 1),
            (4, 5, 6, 7),
            (0, 1, 5, 4),
            (1, 2, 6, 5),
            (2, 3, 7, 6),
            (3, 0, 4, 7),
        ],
        4: [(0, 2, 1), (0, 1, 3), (1, 2, 3), (2, 0, 3)],
        6: [(0, 2, 1), (3, 4, 5), (0, 1, 4, 3), (1, 2, 5, 4), (2, 0, 3, 5)],
    }
    for typ, n in cells:
        for tpl in templates[typ]:
            face = [n[q] for q in tpl]
            key = tuple(sorted(face))
            if key in faces:
                faces[key] = None
            else:
                faces[key] = face
    bnd = []
    for face in faces.values():
        if face is None:
            continue
        points = xyz[np.asarray(face) - 1]
        if np.allclose(points[:, 0], 0):
            tag = 4
        elif np.allclose(points[:, 0], L):
            tag = 5
        elif np.allclose(points[:, 2], 0) or np.allclose(points[:, 2], H):
            tag = 2
        else:
            tag = 3
        bnd.append((2 if len(face) == 3 else 3, tag, face))
    with path.open("w") as f:
        f.write(
            '$MeshFormat\n2.2 0 8\n$EndMeshFormat\n$PhysicalNames\n5\n3 1 "dielectric"\n2 2 "plates_PEC"\n2 3 "sides_PMC"\n2 4 "near"\n2 5 "far"\n$EndPhysicalNames\n'
        )
        f.write(f"$Nodes\n{len(xyz)}\n")
        for i, (x, y, z) in enumerate(xyz, 1):
            f.write(f"{i} {x:.12g} {y:.12g} {z:.12g}\n")
        f.write(f"$EndNodes\n$Elements\n{len(cells) + len(bnd)}\n")
        eid = 0
        for typ, tag, vs in bnd:
            eid += 1
            f.write(f"{eid} {typ} 2 {tag} {tag} " + " ".join(map(str, vs)) + "\n")
        for typ, vs in cells:
            eid += 1
            f.write(f"{eid} {typ} 2 1 1 " + " ".join(map(str, vs)) + "\n")
        f.write("$EndElements\n")
    return len(cells)


def tem_config(order, zref, dense=False):
    driven = {"Samples": [{"Type": "Point", "Freq": [25.0, 112.5, 200.0]}]}
    if dense:
        driven = {
            "Samples": [
                {
                    "Type": "Linear",
                    "MinFreq": 0.25,
                    "MaxFreq": 200.0,
                    "FreqStep": 0.25,
                    "SaveStep": 0,
                }
            ],
            "AdaptiveTol": 1e-7,
            "AdaptiveMaxSamples": 80,
        }
    return {
        "Problem": {"Type": "Driven", "Verbose": 1, "Output": "output/palace"},
        "Model": {"Mesh": "palace.msh", "L0": 1e-6, "Refinement": {"MaxIts": 0}},
        "Domains": {
            "Materials": [{"Attributes": [1], "Permittivity": TEM_EPS, "LossTan": 0.0}]
        },
        "Boundaries": {
            "PEC": {"Attributes": [2]},
            "LumpedPort": [
                {
                    "Index": index,
                    "Direction": "Z",
                    "Excitation": index,
                    "Attributes": [index + 3],
                    "R": zref,
                }
                for index in (1, 2)
            ],
        },
        "Solver": {
            "Order": order,
            "Linear": {
                "Type": "MUMPS",
                "KSPType": "GMRES",
                "Tol": 1e-11,
                "MaxIts": 4,
                "MGMaxLevels": 1,
                "PCMatReal": False,
                "ComplexCoarseSolve": True,
                "EstimatorMaxIts": 0,
                "DivFreeMaxIts": 0,
            },
            "Driven": driven,
        },
    }


def tem_launch(directory, config):
    executable = shutil.which(PALACE_EXECUTABLE or "palace")
    if executable is None:
        raise FileNotFoundError("Set PALACE_EXECUTABLE to a local Palace launcher")
    (directory / "config.json").write_text(json_text(config))
    env = os.environ.copy()
    runtime_lib = Path(executable).resolve().parent.parent / "lib"
    if runtime_lib.is_dir():
        env["LD_LIBRARY_PATH"] = (
            str(runtime_lib) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
        )
    env["OPENBLAS_NUM_THREADS"] = "1"
    with (directory / "solver.log").open("w") as log:
        result = subprocess.run(  # noqa: S603 -- explicit user-selected Palace executable
            [executable, "-np", "1", "-nt", "1", "config.json"],
            cwd=directory,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=False,
        )
    if result.returncode:
        raise RuntimeError(f"Palace exited {result.returncode}; inspect {directory}")
    return tem_validate(
        directory,
        adaptive="AdaptiveTol" in config["Solver"]["Driven"],
        exit_code=result.returncode,
    )


def tem_validate(directory, adaptive=False, exit_code=None):
    report = validate_solution(
        (directory / "solver.log").read_text(),
        estimator_disabled=True,
        exit_code=exit_code,
    )
    assert report.core.status == "passed", report.core
    if adaptive:
        assert len(report.adaptive) == 1
        assert report.adaptive[0].check.status == "passed"
    return report


def tem_read_columns(directory, sources):
    frame = pd.read_csv(directory / "output/palace/port-S.csv")
    frame.columns = frame.columns.str.strip()
    columns = np.empty((len(frame), 2, len(sources)), complex)
    for j, source in enumerate(sources):
        for response in (1, 2):
            amplitude = 10 ** (frame[f"|S[{response}][{source}]| (dB)"].to_numpy() / 20)
            phase = np.deg2rad(frame[f"arg(S[{response}][{source}]) (deg.)"].to_numpy())
            columns[:, response - 1, j] = amplitude * np.exp(1j * phase)
    assert np.isfinite(columns).all()
    return frame.iloc[:, 0].to_numpy() * 1e9, columns


def tem_short_case(kind, order, hx):
    directory = TEM_ROOT / f"{kind}-p{order}-hx{hx:g}"
    if RUN_TEM_CALIBRATION:
        directory.mkdir(parents=True, exist_ok=True)
        cells = tem_write_box(directory / "palace.msh", kind, hx)
        tem_launch(directory, tem_config(order, TEM_ZC))
        f, s = tem_read_columns(directory, (1, 2))
        exact = line_network(f, TEM_ZC, 1e-3 * np.sqrt(TEM_EPS) / light_speed, TEM_ZC).s
        np.savez_compressed(
            directory / "network.npz", f=f, s=s, zref=TEM_ZC, analytic=exact
        )
        manifest = {
            "status": "complete",
            "geometry": kind,
            "order": order,
            "hx_um": hx,
            "cells": cells,
            "exit_code": 0,
            "sha256": hashlib.sha256(
                (directory / "network.npz").read_bytes()
            ).hexdigest(),
            "scope": "ideal TEM verification",
        }
        (directory / "manifest.json").write_text(json_text(manifest))
    if not (directory / "network.npz").exists():
        return None
    manifest = json.loads((directory / "manifest.json").read_text())
    if manifest.get("status") != "complete":
        return None
    tem_validate(directory)
    actual_config = json.loads((directory / "config.json").read_text())
    assert actual_config == tem_config(order, TEM_ZC), "TEM configuration differs"
    if "sha256" in manifest:
        assert (
            hashlib.sha256((directory / "network.npz").read_bytes()).hexdigest()
            == manifest["sha256"]
        )
    with np.load(directory / "network.npz", allow_pickle=False) as saved:
        f, s = saved["f"], saved["s"]
    csv_f, csv_s = tem_read_columns(directory, (1, 2))
    np.testing.assert_array_equal(f, csv_f)
    np.testing.assert_allclose(s, csv_s, rtol=0, atol=1e-14)
    assert s.shape == (3, 2, 2)
    np.testing.assert_allclose(f, np.array([25, 112.5, 200]) * 1e9)
    exact = line_network(f, TEM_ZC, 1e-3 * np.sqrt(TEM_EPS) / light_speed, TEM_ZC).s
    stats = json.loads((directory / "output/palace/palace.json").read_text())["Problem"]
    return {
        "element": kind,
        "order": order,
        "hx_um": hx,
        "unknowns": stats["DegreesOfFreedom"],
        "max complex S error": float(np.max(np.abs(s - exact))),
    }


TEM_PROFILES = [(1, 50.0), (1, 25.0), (1, 12.5), (2, 50.0), (2, 25.0)]
tem_rows = [
    row
    for kind in ("tet", "prism", "hex")
    for order, hx in TEM_PROFILES
    if (row := tem_short_case(kind, order, hx)) is not None
]
tem_table = pd.DataFrame(tem_rows)
if len(tem_table):
    show(tem_table)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for (kind, order), group in tem_table.groupby(["element", "order"]):
        ax.loglog(
            group.hx_um, group["max complex S error"], "o-", label=f"{kind}, p={order}"
        )
    ax.set(
        xlabel="Axial spacing (µm)",
        ylabel="Max absolute complex-S error",
        title="Actual Palace TEM solves against the exact line",
    )
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
if len(tem_table) != 15:
    print("TEM mesh/order verification INCOMPLETE: regenerate missing local archives.")


def tem_dense_case():
    directory = TEM_ROOT / "prism-p2-hx25-L6000-R50-dense"
    if RUN_TEM_CALIBRATION:
        directory.mkdir(parents=True, exist_ok=True)
        tem_write_box(directory / "palace.msh", "prism", 25.0, L=6000.0)
        columns = []
        for source in (1, 2):
            chunk = directory / f"source{source}"
            chunk.mkdir(exist_ok=True)
            shutil.copyfile(directory / "palace.msh", chunk / "palace.msh")
            config = tem_config(2, 50.0, dense=True)
            for port in config["Boundaries"]["LumpedPort"]:
                if port["Index"] != source:
                    port.pop("Excitation")
            tem_launch(chunk, config)
            f, column = tem_read_columns(chunk, (source,))
            columns.append((f, column))
        np.testing.assert_allclose(columns[0][0], columns[1][0], rtol=0, atol=1)
        s = np.concatenate([column for _, column in columns], axis=2)
        np.savez_compressed(directory / "network.npz", f=f, s=s, zref=50.0)
        (directory / "manifest.json").write_text(json_text({"status": "complete"}))
    if not (directory / "network.npz").exists():
        print("Dense TEM/TDR verification INCOMPLETE: no complete local archive.")
        return None
    for source in (1, 2):
        chunk = directory / f"source{source}"
        config = json.loads((chunk / "config.json").read_text())
        assert [
            p["Excitation"]
            for p in config["Boundaries"]["LumpedPort"]
            if "Excitation" in p
        ] == [source]
        expected_config = tem_config(2, 50.0, dense=True)
        for port in expected_config["Boundaries"]["LumpedPort"]:
            if port["Index"] != source:
                port.pop("Excitation")
        assert config == expected_config, "Dense TEM configuration differs"
        tem_validate(chunk, adaptive=True)
    with np.load(directory / "network.npz", allow_pickle=False) as saved:
        f, s = saved["f"], saved["s"]
    assert s.shape == (800, 2, 2)
    assert np.isfinite(s).all()
    np.testing.assert_allclose(f, np.arange(1, 801) * 0.25e9)
    delay = 0.006 * np.sqrt(TEM_EPS) / light_speed
    exact = line_network(f, TEM_ZC, delay, 50.0).s
    t, _, impedance = tdr(f, s[:, 0, 0], 50.0)
    time_resolution = 2 / f[-1]
    plateau = (t > 2.5 * time_resolution) & (t < 2 * delay - 2.5 * time_resolution)
    assert plateau.any()
    impedance_estimate = np.median(impedance[plateau])
    # For this ideal PEC line only, S21(0)=1 exactly; reflection uses our general DC rule.
    through = rf.Network(
        f=np.r_[0.0, f], s=np.r_[1.0, s[:, 1, 0]][:, None, None], z0=50.0, f_unit="Hz"
    )
    ti, impulse = through.impulse_response(window="hamming", bandpass=False)
    delay_estimate = ti[np.argmax(np.abs(impulse))]
    metrics = {
        "max complex S error": float(np.max(np.abs(s - exact))),
        "known impedance (ohm)": TEM_ZC,
        "recovered impedance (ohm)": impedance_estimate,
        "known delay (ps)": delay * 1e12,
        "recovered delay (ps)": delay_estimate * 1e12,
        "time resolution (ps)": time_resolution * 1e12,
    }
    show(pd.Series(metrics).to_frame("Ideal-line EM/TDR check"))
    assert metrics["max complex S error"] < 1e-3
    assert abs(impedance_estimate / TEM_ZC - 1) < 0.02
    assert abs(delay_estimate - delay) < time_resolution
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(t * 1e12, impedance, label="Palace → TDR")
    axes[0].axhline(TEM_ZC, color="k", ls=":", label="Exact Zc")
    axes[0].axvspan(
        t[plateau].min() * 1e12,
        t[plateau].max() * 1e12,
        alpha=0.1,
        color="C0",
        label="Plateau measurement gate",
    )
    axes[0].set(
        xlim=(-10, 110), ylim=(40, 110), xlabel="Time (ps)", ylabel="Impedance (ohm)"
    )
    axes[1].plot(ti * 1e12, impulse, label="Palace transmitted impulse")
    axes[1].axvline(delay * 1e12, color="k", ls=":", label="Exact one-way delay")
    axes[1].set(xlim=(0, 90), xlabel="Time (ps)", ylabel="Impulse response")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("Ideal TEM EM calibration — independent of IHP convergence")
    fig.tight_layout()
    return metrics


tem_tdr_metrics = tem_dense_case()

## 16. Align the mesh with the interconnect

The recovery studies show that making every tetrahedron smaller is expensive,
and that the first-order field approximation is inadequate. The next experiment
uses a triangular cross-section swept into prisms along the long traces. This
lets us choose **axial spacing**, **conductor-corner spacing**, and **far-region
spacing** separately. These controls change the finite-element approximation;
all layer heights, dielectric/conductor coefficients, terminal sheets, and outer
boundaries retain their declared physical meaning. Section 15 verified prism
handling for a uniform TEM field. The real conductor corners still need their
own refinement study.

The geometry here has rectangular conductors and disjoint, flat dielectric
intervals. The helper below is specialized to that geometry, and takes its
material names, layer roles, and heights from a supplied stack. It does not use
IHP layer/material names. `MeshResult`, `PalacePort`, and `write_config` connect
this mesh to gsim's ordinary material/port configuration code. The IHP selection
and overrides remain caller inputs. The laboratory uses a local Palace runtime.

Three details matter:

1. **Conductor interiors.** A surface-impedance boundary models the exterior
   fields without resolving fields inside the metal. The original mesher leaves
   dielectric-filled, closed conductor interiors behind those boundaries. Our
   pilot removed these unexcited enclosed regions, retaining exactly the same
   conductor surfaces and coefficients. Two independently solved columns (25
   and 200 GHz, source p1) changed by less than $1.4\times10^{-11}$. This is an
   empirical equivalence check at those points, not a general theorem about all
   meshes. `exterior_only=False` retains the filler for further checks. Do not
   change Palace's `External` flag: that would change its effective-thickness
   convention and the physical boundary model.
2. **Symmetry.** Optional mirroring constructs a reflection-symmetric *mesh*.
   Every requested source is still solved independently on the complete domain.
   No S-parameter matrix is symmetrized. Exact mesh symmetry can suppress
   baseline asymmetry, but cannot establish propagation or gap accuracy.
3. **Paired grids.** Reuse the same cross-section for intact/disrupted/moved-slot
   cases. Include both slot positions in the axial landmarks. The physical hole
   activates dielectric cells previously enclosed by the return metal; geometry
   changes are confined to the intended slot. Refining corners in this cross-
   section does not refine the two axial slot ends. The axial stations must be
   refined separately before accepting a three-dimensional gap response.

**Pilot evidence, still incomplete:** the aligned second-order source-p1 column
changes by **0.00446 at 25 GHz** and **0.01030 at 200 GHz** between the original
100 µm axial / 2 µm corner grid and the 75 µm axial / 0.25 µm corner grid. Both
solves pass the linear checks. These are only two columns of different meshes,
not full-network convergence. The completed 0.125 µm corner study changes that column by **0.00097 at 25 GHz** and **0.00383 at 200 GHz**; all its linear solves pass. Independent axial and order checks are now running.
The 0.01 full mixed-mode target and resolved-conversion/TDR gates remain unchanged.

The code includes controls to run complete direct-anchor or dense four-port
networks, with manifests and hashes. Leave `RUN_ALIGNED=False` to inspect the
implementation without launching a simulation. The listed meshes are candidates,
not accepted references. Run the independent refinement comparisons before
using their gap spectra quantitatively.


**PDK separation check:** `CHECK_ALIGNED_SETUP=True` builds the synthetic
stack below with renamed metals/dielectrics, different heights, and one finite
dielectric region. This checks the new mesh/configuration path. It is a setup
fixture, not another device-validation example. The recorded check passed,
including a Palace configuration dry run. The new IHP mesh/configuration also
matches the prototype's nodes, element roles, and physical coefficients exactly
(up to physical-group identifiers and internal-interface bookkeeping).


`axial_end_um` adds geometric grading on both sides of terminal and slot-end
planes over `axial_end_extent_um`. For example, compare 2, 1, and 0.5 µm at
fixed transverse mesh, then refine the transverse mesh independently. The pilot
columns above used **no axial end grading**; the graded candidate has not been
validated. Reducing the nominal axial wavelength spacing alone can leave a
slot's first two cells almost unchanged, so that comparison would be weak
evidence about its localized reflection. The same graded landmarks are used in
every paired geometry.


**Dense-sweep solver check:** direct anchors can miss a nearly singular
unshifted real preconditioner. A small IHP pipeline fixture hit repeated GMRES
failures near 10.8 GHz with that choice; its incomplete dense data are rejected.
The laboratory therefore uses `PCMatShifted=True` with the real preconditioner:
the physical complex Maxwell operator remains unshifted. At 25 and 200 GHz the
large pilot's shifted/unshifted independently solved columns agree within
$3.1\times10^{-9}$ after both residual checks pass. Positive shifting requires
more Krylov iterations (460 at 200 GHz in that probe); the iteration limit is
explicit, and any failed main solve still rejects the complete network. The shifted small-fixture full-band check has now completed all 800 frequencies
and all four independent excitations with every linear/adaptive check passing.
This is a 200 µm IHP pipeline fixture, not a substitute for the 6 mm device.
The large-model dense and spatial-convergence checks remain open.


### 16.1 Cross-section mesh

Set the execution and mesh controls here. The cross-section generator fragments
the dielectric intervals and conductor/terminal boundaries before triangulation.
Its optional reflection step mirrors mesh triangles, not simulated responses.

In [ ]:
# Independent axial/corner/field-order candidates. All dimensions are µm.
RUN_ALIGNED = False
ALIGNED_DENSE = False  # True: 800 points, 0.25-200 GHz; False: direct anchors
ALIGNED_PROFILE = {
    "order": 2,
    "axial_um": 75.0,
    "corner_um": 0.125,
    "far_um": 40.0,
    "air_um": 150.0,
    "mirror": True,
    "exterior_only": True,
    "axial_end_um": 1.0,
    "axial_end_extent_um": 20.0,
    "positive_preconditioner": True,
    "linear_max_iterations": 1000,
    "linear_restart": 120,
}
ALIGNED_ANCHORS_GHZ = (25.0, 112.5, 200.0)
ALIGNED_SLOT_CENTRES = (GEO.gap_x, GEO.gap_x + 1200.0)


import itertools
import json
from pathlib import Path

import gmsh
import numpy as np


def aligned_cross_section(
    g,
    signal_z,
    return_z,
    dielectrics,
    margin,
    corner,
    hfar,
    interior_alignment=True,
    point_sizing=True,
    mirror=False,
):
    W, w, s = g["ground_width"], g["width"], g["separation"]
    gy, gw = g["gap_y"], g["gap_width"]
    zmin = min(d["zmin"] for d in dielectrics)
    zmax = max(d["zmax"] for d in dielectrics)
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("cross_section")
    occ = gmsh.model.occ
    rects = [
        occ.addRectangle(
            -W / 2 - margin, zmin - margin, 0, W + 2 * margin, zmax - zmin + 2 * margin
        )
    ]
    for d in dielectrics:
        rects.append(occ.addRectangle(-W / 2, d["zmin"], 0, W, d["zmax"] - d["zmin"]))  # noqa: PERF401 -- explicit CAD entities
    cuts = np.unique(
        [
            -W / 2,
            W / 2,
            gy - gw / 2,
            gy + gw / 2,
            -gy - gw / 2,
            -gy + gw / 2,
            *[c + q for c in [-s / 2, s / 2] for q in [-w / 2, w / 2]],
        ]
    )
    for a, b in itertools.pairwise(cuts):
        rects.append(
            occ.addRectangle(a, return_z[0], 0, b - a, return_z[1] - return_z[0])
        )
    for center in [-s / 2, s / 2]:
        rects.append(
            occ.addRectangle(
                center - w / 2, signal_z[0], 0, w, signal_z[1] - signal_z[0]
            )
        )
        rects.append(
            occ.addRectangle(
                center - w / 2, return_z[1], 0, w, signal_z[0] - return_z[1]
            )
        )
    for center in [gy, -gy]:
        rects.append(  # noqa: PERF401 -- explicit CAD entities
            occ.addRectangle(
                center - gw / 2, return_z[0], 0, gw, return_z[1] - return_z[0]
            )
        )
    if mirror:
        rects.append(
            occ.addRectangle(
                0, zmin - margin, 0, W / 2 + margin, zmax - zmin + 2 * margin
            )
        )
    occ.fragment([(2, rects[0])], [(2, t) for t in rects[1:]])
    occ.synchronize()
    # Small edges/corners near conductors; graded outer air/substrate.
    gmsh.option.setNumber("Mesh.MeshSizeMin", corner * 0.25)
    gmsh.option.setNumber("Mesh.MeshSizeMax", hfar)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", int(point_sizing))
    gmsh.option.setNumber("Mesh.Algorithm", 6)
    gmsh.option.setNumber("General.NumThreads", 1)
    gmsh.model.mesh.setSize(gmsh.model.getEntities(0), hfar)
    near = []
    for dim, tag in gmsh.model.getEntities(0):
        y, z, _ = gmsh.model.getValue(dim, tag, [])
        if abs(y) <= W / 2 + 1e-8 and return_z[0] - 1e-8 <= z <= signal_z[1] + 1e-8:
            near.append((dim, tag))
    gmsh.model.mesh.setSize(near, corner)
    distance = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(distance, "PointsList", [tag for _, tag in near])
    threshold = gmsh.model.mesh.field.add("Threshold")
    for key, value in [
        ("InField", distance),
        ("SizeMin", corner),
        ("SizeMax", hfar),
        ("DistMin", corner),
        ("DistMax", 16.0),
    ]:
        gmsh.model.mesh.field.setNumber(threshold, key, value)
    gmsh.model.mesh.field.setAsBackgroundMesh(threshold)
    # Align triangles within thin rectangular conductor interiors, instead of
    # forcing isotropic elements along a 0.42 um return layer.
    for dim, tag in gmsh.model.getEntities(2):
        xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
        thin = abs(ymin - return_z[0]) < 1e-6 and abs(ymax - return_z[1]) < 1e-6
        trace = (
            abs(ymin - signal_z[0]) < 1e-6
            and abs(ymax - signal_z[1]) < 1e-6
            and xmax - xmin <= w + 1e-6
        )
        if not interior_alignment or not (thin or trace):
            continue
        boundary = gmsh.model.getBoundary([(dim, tag)], oriented=False)
        if len(boundary) != 4:
            continue
        for _, edge in boundary:
            x0, y0, _, x1, y1, _ = gmsh.model.getBoundingBox(1, edge)
            length = max(x1 - x0, y1 - y0)
            target = 2 * corner if x1 - x0 > y1 - y0 else corner
            gmsh.model.mesh.setTransfiniteCurve(
                edge, max(2, int(np.ceil(length / target - 1e-7)) + 1)
            )
        gmsh.model.mesh.setTransfiniteSurface(tag)
    gmsh.model.mesh.generate(2)
    tags, coords, _ = gmsh.model.mesh.getNodes()
    coords = np.asarray(coords).reshape(-1, 3)
    order = np.argsort(tags)
    tags = np.asarray(tags)[order]
    yz = coords[order, :2]
    types, _, node_lists = gmsh.model.mesh.getElements(2)
    triangles = []
    for typ, nodes in zip(types, node_lists, strict=True):
        if typ != 2:
            raise ValueError("Expected linear triangles")
        triangles.extend(np.searchsorted(tags, np.asarray(nodes).reshape(-1, 3)))
    tri = np.asarray(triangles, dtype=np.int64)
    points = yz[tri]
    a = points[:, 1] - points[:, 0]
    b = points[:, 2] - points[:, 0]
    cross = a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    assert (cross > 0).all()
    gmsh.finalize()
    if mirror:
        positive = tri[np.min(yz[tri, 0], axis=1) >= -1e-9]
        used = np.unique(positive)
        points = yz[used].copy()
        points[np.abs(points[:, 0]) < 1e-9, 0] = 0.0
        triangles = np.searchsorted(used, positive)
        offaxis = np.flatnonzero(points[:, 0] > 0.0)
        reflection = np.arange(len(points))
        reflection[offaxis] = np.arange(len(points), len(points) + len(offaxis))
        reflected = points[offaxis].copy()
        reflected[:, 0] *= -1
        yz = np.concatenate([points, reflected])
        tri = np.concatenate([triangles, reflection[triangles[:, [0, 2, 1]]]])
        q = yz[tri]
        det = (q[:, 1, 0] - q[:, 0, 0]) * (q[:, 2, 1] - q[:, 0, 1]) - (
            q[:, 1, 1] - q[:, 0, 1]
        ) * (q[:, 2, 0] - q[:, 0, 0])
        assert np.min(det) > 0
        expected = (W + 2 * margin) * (zmax - zmin + 2 * margin)
        np.testing.assert_allclose(np.sum(det) / 2, expected, rtol=1e-10)
    return yz, tri

### 16.2 Axial stations and prism assembly

The station builder resolves terminal and slot-end planes, with optional local
grading. Sweeping the section gives a conforming prism mesh. Physical groups
identify dielectric regions, conductor surfaces, ports, and the outer boundary.

In [ ]:
def aligned_axial_stations(
    length,
    margin,
    gap_length,
    slot_centres,
    max_step,
    end_step=None,
    end_extent=20.0,
    growth=2.5,
):
    """Refine actual conductor/slot ends; use identical landmarks in paired cases."""
    if min(length, margin, gap_length, max_step) <= 0:
        raise ValueError("Use positive dimensions and axial spacing")
    features = np.unique(
        [
            0.0,
            length,
            *[x + q for x in slot_centres for q in (-gap_length / 2, gap_length / 2)],
        ]
    )
    if np.any(features < 0) or np.any(features > length):
        raise ValueError("Slot landmarks must remain inside the line")
    knots = [-margin, length + margin, *features]
    if end_step is not None:
        if end_step <= 0 or end_extent <= 0 or growth <= 1:
            raise ValueError(
                "Use positive end spacing/extent and growth greater than one"
            )
        neighbours = np.r_[-margin, features, length + margin]
        for i, feature in enumerate(features, 1):
            for direction, neighbour in [
                (-1, neighbours[i - 1]),
                (1, neighbours[i + 1]),
            ]:
                limit = min(end_extent, abs(feature - neighbour) / 2)
                distance, step = 0.0, min(end_step, max_step)
                while distance + step < limit:
                    distance += step
                    knots.append(feature + direction * distance)
                    step = min(growth * step, max_step)
                if limit > 0:
                    knots.append(feature + direction * limit)
    knots = np.unique(np.round(knots, 12))
    xs = np.concatenate(
        [
            np.linspace(a, b, int(np.ceil((b - a) / max_step)) + 1)[:-1]
            for a, b in itertools.pairwise(knots)
        ]
        + [knots[-1:]]
    )
    assert np.all(np.diff(xs) > 0)
    assert np.max(np.diff(xs)) <= max_step * (1 + 1e-12)
    return xs


def make_aligned_mesh(
    path,
    g,
    signal_z,
    return_z,
    dielectrics,
    margin=150.0,
    hx=150.0,
    corner=2.0,
    hfar=40.0,
    gap=False,
    section=None,
    axial_stations=None,
    exclude_interiors=False,
    interior_alignment=True,
    point_sizing=True,
    mirror=False,
    signal_key="signal",
    return_key="return",
    slot_centres=None,
    axial_end_um=None,
    axial_end_extent_um=20.0,
):
    yz, tri = (
        aligned_cross_section(
            g,
            signal_z,
            return_z,
            dielectrics,
            margin,
            corner,
            hfar,
            interior_alignment,
            point_sizing,
            mirror,
        )
        if section is None
        else section
    )
    L, W, w, s = g["length"], g["ground_width"], g["width"], g["separation"]
    gx, gl, gy, gw = g["gap_x"], g["gap_length"], g["gap_y"], g["gap_width"]
    centres = (gx,) if slot_centres is None else tuple(slot_centres)
    xs = aligned_axial_stations(
        L,
        margin,
        gl,
        centres,
        hx,
        end_step=axial_end_um,
        end_extent=axial_end_extent_um,
    )
    if axial_stations is not None:
        xs = np.asarray(axial_stations, float)
    for required in [-margin, 0, gx - gl / 2, gx + gl / 2, L, L + margin]:
        if np.min(np.abs(xs - required)) > 1e-8:
            raise ValueError("Axial stations omit a device/slot boundary")
    nx = len(xs) - 1
    n2 = len(yz)
    nt = len(tri)
    Y, Z = yz[tri].mean(axis=1).T
    X = (xs[:-1] + xs[1:]) / 2
    X = X[:, None]
    Y = Y[None, :]
    Z = Z[None, :]
    body = (X > 0) & (X < L) & (np.abs(Y) < W / 2)
    ids = {
        name: i + 1
        for i, name in enumerate(sorted({"air", *[d["material"] for d in dielectrics]}))
    }
    base = len(ids)
    outer = base + 9
    material = np.full((nx, nt), ids["air"], dtype=np.int64)
    for d in dielectrics:
        material[body & (d["zmin"] < Z) & (d["zmax"] > Z)] = ids[d["material"]]
    conductor = np.zeros_like(material)
    ret = body & (return_z[0] < Z) & (return_z[1] > Z)
    if gap:
        ret &= ~((abs(X - gx) < gl / 2) & (abs(Y - gy) < gw / 2))
    conductor[ret] = 1
    conductor[
        (X > 0)
        & (X < L)
        & ((abs(Y - s / 2) < w / 2) | (abs(Y + s / 2) < w / 2))
        & (signal_z[0] < Z)
        & (signal_z[1] > Z)
    ] = 2
    node = np.arange(1, (nx + 1) * n2 + 1).reshape(nx + 1, n2)
    prisms = np.concatenate([node[:-1, tri], node[1:, tri]], axis=-1).reshape(-1, 6)
    coords = np.concatenate([np.c_[np.full(n2, x), yz] for x in xs])
    interfaces = {
        (a, b): base + 10 + i
        for i, (a, b) in enumerate(
            (a, b) for a in ids.values() for b in ids.values() if a < b
        )
    }

    def labels(cl, cr, ml, mr, zface=False):
        a = np.zeros(cl.shape, dtype=np.int64)
        a[(cl == -1) | (cr == -1)] = outer
        for pair, tag in interfaces.items():
            a[
                ((ml == pair[0]) & (mr == pair[1]))
                | ((ml == pair[1]) & (mr == pair[0]))
            ] = tag
        for code, xy, side in [(1, base + 1, base + 2), (2, base + 3, base + 4)]:
            a[(cl != cr) & ((cl == code) | (cr == code))] = xy if zface else side
        return a

    triangles = []
    quads = []
    for i, x in enumerate(xs):
        cl = conductor[i - 1] if i else np.full(nt, -1)
        cr = conductor[i] if i < nx else np.full(nt, -1)
        ml = material[i - 1] if i else np.full(nt, -1)
        mr = material[i] if i < nx else np.full(nt, -1)
        lab = labels(cl, cr, ml, mr)
        if abs(x) < 1e-8 or abs(x - L) < 1e-8:
            for center, near, far in [
                (s / 2, base + 5, base + 6),
                (-s / 2, base + 7, base + 8),
            ]:
                port = (
                    (abs(Y[0] - center) < w / 2)
                    & (Z[0] > return_z[1])
                    & (Z[0] < signal_z[0])
                )
                lab[port] = near if abs(x) < 1e-8 else far
        for j in np.flatnonzero(lab):
            triangles.append((int(lab[j]), node[i, tri[j]].tolist()))  # noqa: PERF401 -- explicit faces
    edges = {}
    for j, t in enumerate(tri):
        for a, b in [(t[0], t[1]), (t[1], t[2]), (t[2], t[0])]:
            edges.setdefault(tuple(sorted((int(a), int(b)))), []).append(j)
    for (a, b), adj in edges.items():
        assert len(adj) <= 2
        left = adj[0]
        right = adj[1] if len(adj) == 2 else None
        cl = conductor[:, left]
        ml = material[:, left]
        cr = conductor[:, right] if right is not None else np.full(nx, -1)
        mr = material[:, right] if right is not None else np.full(nx, -1)
        lab = labels(cl, cr, ml, mr, zface=abs(yz[a, 1] - yz[b, 1]) < 1e-8)
        for i in np.flatnonzero(lab):
            quads.append(  # noqa: PERF401 -- explicit faces
                (int(lab[i]), [node[i, a], node[i, b], node[i + 1, b], node[i + 1, a]])
            )
    volume_labels = material.ravel()
    removed_count = 0
    if exclude_interiors:
        keep = conductor.ravel() == 0
        removed_count = int((~keep).sum())
        prisms = prisms[keep]
        volume_labels = volume_labels[keep]
        used = np.unique(
            np.concatenate(
                [
                    prisms.ravel(),
                    np.asarray(
                        [n for _, vs in triangles + quads for n in vs], dtype=np.int64
                    ),
                ]
            )
        )
        remap = np.zeros(len(coords) + 1, dtype=np.int64)
        remap[used] = np.arange(1, len(used) + 1)
        coords = coords[used - 1]
        prisms = remap[prisms]
        triangles = [(tag, remap[np.asarray(vs)].tolist()) for tag, vs in triangles]
        quads = [(tag, remap[np.asarray(vs)].tolist()) for tag, vs in quads]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    names = {tag: name for name, tag in ids.items()}
    names.update(
        {
            base + 1: f"{return_key}_xy",
            base + 2: f"{return_key}_z",
            base + 3: f"{signal_key}_xy",
            base + 4: f"{signal_key}_z",
            **{base + 5 + i: f"P{i + 1}" for i in range(4)},
            outer: "outer",
        }
    )
    for pair, tag in interfaces.items():
        names[tag] = "interface_" + names[pair[0]] + "_" + names[pair[1]]
    groups = {
        "volumes": {name: {"phys_group": tag} for name, tag in ids.items()},
        "conductor_surfaces": {
            names[tag]: {"phys_group": tag} for tag in range(base + 1, base + 5)
        },
        "port_surfaces": {
            f"P{i + 1}": {"phys_group": base + 5 + i, "type": "via"} for i in range(4)
        },
        "boundary_surfaces": {"absorbing": {"phys_group": outer}},
    }
    with path.open("w") as f:
        f.write(f"$MeshFormat\n2.2 0 8\n$EndMeshFormat\n$PhysicalNames\n{len(names)}\n")
        for tag, name in names.items():
            f.write(f'{3 if tag <= base else 2} {tag} "{name}"\n')
        f.write(f"$EndPhysicalNames\n$Nodes\n{len(coords)}\n")
        for i, (x, y, z) in enumerate(coords, 1):
            f.write(f"{i} {x:.12g} {y:.12g} {z:.12g}\n")
        f.write(f"$EndNodes\n$Elements\n{len(triangles) + len(quads) + len(prisms)}\n")
        eid = 0
        for typ, entries in [
            (2, triangles),
            (3, quads),
            (6, zip(volume_labels, prisms, strict=True)),
        ]:
            for tag, q in entries:
                eid += 1
                f.write(f"{eid} {typ} 2 {tag} {tag} " + " ".join(map(str, q)) + "\n")
        f.write("$EndElements\n")
    return {
        "groups": groups,
        "material_ids": ids,
        "mirror": mirror,
        "axial_end_um": axial_end_um,
        "axial_end_extent_um": axial_end_extent_um,
        "nodes": len(coords),
        "prisms": len(prisms),
        "exterior_only": exclude_interiors,
        "inactive_conductor_cells_removed": removed_count,
        "interior_alignment": interior_alignment,
        "point_sizing": point_sizing,
        "cross_section_nodes": n2,
        "cross_section_triangles": nt,
        "nx": nx,
        "hx_um": hx,
        "corner_um": corner,
        "hfar_um": hfar,
        "gap": gap,
        "mesh_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

### 16.3 Configure from a supplied layer stack

This adapter obtains conductor heights/materials and dielectric intervals from
its arguments. It writes a `MeshResult` and uses gsim to generate the material
and port coefficients. Advanced linear controls are explicit in the saved JSON.

In [ ]:
import copy
import json
from pathlib import Path

import numpy as np

from gsim.palace.mesh.config_generator import write_config
from gsim.palace.mesh.generator import MeshResult
from gsim.palace.models import DrivenConfig, SolverConfig
from gsim.palace.ports.config import PalacePort, PortGeometry


def aligned_config(
    stack,
    geometry,
    signal_key,
    return_key,
    directory,
    *,
    profile,
    overrides=None,
    gap=False,
    section=None,
    slot_centres=None,
    frequencies_ghz=(25.0, 112.5, 200.0),
    dense=False,
):
    """Geometry and physical coefficients come from caller-supplied stack/roles."""
    if signal_key == return_key or not {signal_key, return_key} <= stack.layers.keys():
        raise ValueError("Select two distinct conductor layers")
    stack = copy.deepcopy(stack)
    for material, properties in (overrides or {}).items():
        stack.materials.setdefault(material, {}).update(properties)
    sig, ret = stack.layers[signal_key], stack.layers[return_key]
    if sig.zmin <= ret.zmax:
        raise ValueError("Signal must lie above return")
    for layer in [sig, ret]:
        if not stack.materials.get(layer.material, {}).get("conductivity", 0) > 0:
            raise ValueError("Conductor conductivity must be explicit")
    ds = sorted(stack.dielectrics, key=lambda d: d["zmin"])
    for a, b in itertools.pairwise(ds):
        if a["zmax"] > b["zmin"] + 1e-9:
            raise ValueError(
                "This aligned mesher requires disjoint flat dielectric intervals"
            )
    if not ds:
        raise ValueError("Supply explicit finite dielectric regions")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    summary = make_aligned_mesh(
        directory / "palace.msh",
        geometry,
        (sig.zmin, sig.zmax),
        (ret.zmin, ret.zmax),
        stack.dielectrics,
        margin=profile["air_um"],
        hx=profile["axial_um"],
        corner=profile["corner_um"],
        hfar=profile["far_um"],
        gap=gap,
        section=section,
        exclude_interiors=profile["exterior_only"],
        interior_alignment=False,
        point_sizing=False,
        mirror=profile["mirror"],
        signal_key=signal_key,
        return_key=return_key,
        slot_centres=slot_centres,
        axial_end_um=profile.get("axial_end_um"),
        axial_end_extent_um=profile.get("axial_end_extent_um", 20.0),
    )
    ports = []
    port_info = []
    L, w, s = geometry["length"], geometry["width"], geometry["separation"]
    for i, (name, x, y) in enumerate(
        [("p1", 0.0, s / 2), ("p3", L, s / 2), ("p2", 0.0, -s / 2), ("p4", L, -s / 2)],
        1,
    ):
        ports.append(
            PalacePort(
                name=name,
                geometry=PortGeometry.INTERLAYER,
                center=(x, y),
                width=w,
                orientation=180.0 if x == 0 else 0.0,
                zmin=ret.zmax,
                zmax=sig.zmin,
                from_layer=return_key,
                to_layer=signal_key,
                impedance=50.0,
                excited=True,
            )
        )
        port_info.append(
            {
                "portnumber": i,
                "name": name,
                "Z0": 50.0,
                "type": "via",
                "direction": "Z",
                "length": sig.zmin - ret.zmax,
                "width": w,
                "xmin": x,
                "xmax": x,
                "ymin": y - w / 2,
                "ymax": y + w / 2,
                "zmin": ret.zmax,
                "zmax": sig.zmin,
            }
        )
    mesh = MeshResult(
        mesh_path=directory / "palace.msh",
        port_info=port_info,
        mesh_stats={"nodes": summary["nodes"], "prisms": summary["prisms"]},
        groups=summary["groups"],
        output_dir=directory,
        model_name="palace",
        fmax=200e9,
    )
    cfgpath = write_config(
        mesh,
        stack,
        ports,
        driven_config=DrivenConfig(
            fmin=0.25e9 if dense else min(frequencies_ghz) * 1e9,
            fmax=200e9 if dense else max(frequencies_ghz) * 1e9,
            num_points=800 if dense else len(frequencies_ghz),
            adaptive_tol=1e-5 if dense else 0.0,
            adaptive_max_samples=80,
        ),
        numerical_config=SolverConfig(order=profile["order"]),
    )
    cfg = json.loads(cfgpath.read_text())
    cfg["Solver"]["Linear"].update(
        Type="MUMPS",
        KSPType="GMRES",
        Tol=1e-9,
        MaxIts=profile.get("linear_max_iterations", 1000),
        MGMaxLevels=1,
        EstimatorMaxIts=0,
        DivFreeMaxIts=0,
        PCMatReal=True,
        ComplexCoarseSolve=False,
        MaxSize=profile.get("linear_restart", 120),
        PCMatShifted=profile.get("positive_preconditioner", True),
    )
    if not dense:
        cfg["Solver"]["Driven"] = {
            "Samples": [{"Type": "Point", "Freq": list(frequencies_ghz)}]
        }
    cfgpath.write_text(json.dumps(cfg, indent=2))
    (directory / "mesh-summary.json").write_text(json.dumps(summary, indent=2))
    return mesh, cfg, summary

### 16.4 Read and preserve complete four-port experiments

Read every independently solved column, apply the declared port permutation,
and reject missing entries or failed linear/adaptive solves. A completed archive
contains the full network, inputs, solver evidence, and hashes.

In [ ]:
def aligned_read_network(directory):
    frame = pd.read_csv(directory / "output/palace/port-S.csv")
    frame.columns = frame.columns.str.strip()
    s = np.empty((len(frame), 4, 4), complex)
    for source in range(1, 5):
        for response in range(1, 5):
            magnitude = frame[f"|S[{response}][{source}]| (dB)"].to_numpy()
            phase = frame[f"arg(S[{response}][{source}]) (deg.)"].to_numpy()
            s[:, response - 1, source - 1] = 10 ** (magnitude / 20) * np.exp(
                1j * np.deg2rad(phase)
            )
    if not np.isfinite(s).all():
        raise ValueError("Missing/nonfinite independently solved four-port response")
    # Native order p1,p3,p2,p4 -> canonical near A, near B, far A, far B.
    idx = [0, 2, 1, 3]
    net = rf.Network(
        f=frame.iloc[:, 0].to_numpy() * 1e9,
        s=s[:, idx][:, :, idx],
        z0=Z_SE,
        f_unit="Hz",
    )
    net.port_names = list(PORTS)
    check_network(net)
    return net


def aligned_run_case(
    label,
    geometry,
    stack,
    signal_key,
    return_key,
    *,
    gap,
    profile,
    overrides=None,
    section=None,
    slot_centres=None,
    dense=False,
    anchors_ghz=(25.0, 112.5, 200.0),
    provenance=None,
):
    """Run/reload one complete network; no missing column or failed solve is accepted."""
    profile = {
        "axial_end_um": None,
        "axial_end_extent_um": 20.0,
        "positive_preconditioner": True,
        "linear_max_iterations": 1000,
        "linear_restart": 120,
        **profile,
    }
    spec = {
        "schema": 1,
        "geometry_um": asdict(geometry),
        "gap": gap,
        "profile": profile,
        "layers": {
            k: stack.layers[k].model_dump(mode="json") for k in (signal_key, return_key)
        },
        "dielectrics": stack.dielectrics,
        "materials": stack.materials,
        "material_overrides": overrides or {},
        "signal_key": signal_key,
        "return_key": return_key,
        "slot_centres": slot_centres,
        "dense": dense,
        "anchors_ghz": anchors_ghz,
        "palace_version": PALACE_VERSION,
        "versions": VERSIONS,
        "gsim_revision": GSIM_REVISION,
        "provenance": provenance or {},
        "port_names_native": ["p1", "p3", "p2", "p4"],
        "reference_ohm": Z_SE,
        "reference_planes_um": [0.0, geometry.length],
    }
    if section is not None:
        spec["section_sha256"] = hashlib.sha256(
            np.ascontiguousarray(section[0], dtype=np.float64).tobytes()
            + np.ascontiguousarray(section[1], dtype=np.int64).tobytes()
        ).hexdigest()
    signature = hashlib.sha256(json_text(spec).encode()).hexdigest()[:16]
    directory = OUTPUT / f"aligned-{label}-{signature}"
    manifest_path, data_path = directory / "manifest.json", directory / "network.npz"
    if manifest_path.exists() and data_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("status") == "complete":
            assert json_text(manifest["spec"]) == json_text(spec)
            assert (
                hashlib.sha256(data_path.read_bytes()).hexdigest() == manifest["sha256"]
            )
            with np.load(data_path, allow_pickle=False) as saved:
                net = rf.Network(
                    f=saved["f"], s=saved["s"], z0=saved["z0"], f_unit="Hz"
                )
                net.port_names = saved["ports"].tolist()
            check_network(net)
            return net
    if not RUN_ALIGNED:
        return None
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {"status": "incomplete", "spec": spec}
    manifest_path.write_text(json_text(manifest))
    try:
        _mesh, config, summary = aligned_config(
            stack,
            asdict(geometry),
            signal_key,
            return_key,
            directory,
            profile=profile,
            overrides=overrides,
            gap=gap,
            section=section,
            slot_centres=slot_centres,
            frequencies_ghz=anchors_ghz,
            dense=dense,
        )
        validate_generated_config(config)
        manifest["inputs"] = {
            name: hashlib.sha256((directory / name).read_bytes()).hexdigest()
            for name in ("palace.msh", "config.json", "port_information.json")
        }
        manifest["mesh"] = summary
        manifest_path.write_text(json_text(manifest))
        executable = shutil.which(PALACE_EXECUTABLE or "palace")
        if executable is None:
            raise FileNotFoundError("Set PALACE_EXECUTABLE to a local Palace launcher")  # noqa: TRY301 -- record failed experiment
        env = os.environ.copy()
        runtime_lib = Path(executable).resolve().parent.parent / "lib"
        if runtime_lib.is_dir():
            env["LD_LIBRARY_PATH"] = (
                str(runtime_lib) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
            )
        env["OPENBLAS_NUM_THREADS"] = "1"
        with (directory / "solver.log").open("w") as log:
            result = subprocess.run(  # noqa: S603 -- explicit user-selected Palace executable
                [executable, "-np", str(LOCAL_PROCESSES), "-nt", "2", "config.json"],
                cwd=directory,
                env=env,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=False,
            )
        manifest["exit_code"] = result.returncode
        if result.returncode:
            raise RuntimeError(  # noqa: TRY301 -- recorded by this experiment manifest
                f"Palace exited {result.returncode}; inspect {directory}"
            )
        evidence = validate_solution(
            (directory / "solver.log").read_text(),
            estimator_disabled=True,
            expected_excitations=[1, 2, 3, 4] if dense else [],
            exit_code=result.returncode,
        )
        manifest["solver_evidence"] = asdict(evidence)
        assert evidence.core.status == "passed", evidence.core
        if dense:
            assert len(evidence.adaptive) == 4
            assert all(a.check.status == "passed" for a in evidence.adaptive)
        net = aligned_read_network(directory)
        expected_f = (
            np.arange(1, 801) * 0.25e9 if dense else np.asarray(anchors_ghz) * 1e9
        )
        np.testing.assert_allclose(net.f, expected_f, rtol=0, atol=1)
        np.savez_compressed(
            data_path, f=net.f, s=net.s, z0=net.z0, ports=np.asarray(net.port_names)
        )
        net.write_touchstone(filename=str(directory / "network"))
        manifest.update(
            status="complete", sha256=hashlib.sha256(data_path.read_bytes()).hexdigest()
        )
        manifest_path.write_text(json_text(manifest))
    except Exception as exc:
        manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        manifest_path.write_text(json_text(manifest))
        raise
    return net

### 16.5 Run paired geometries

All three geometries share a cross-section and both slot-end landmark sets.
Changing `ALIGNED_PROFILE` selects another discretization and therefore a
different archive. Compare complete networks at matching frequencies; direct
anchors alone cannot establish dense-spectrum or TDR convergence.

In [ ]:
aligned_networks = {}
if RUN_ALIGNED:
    aligned_stack = build_palace_layer_stack(resolved)
    signal, ground = aligned_stack.layers[SIGNAL_KEY], aligned_stack.layers[RETURN_KEY]
    shared_section = aligned_cross_section(
        asdict(GEO),
        (signal.zmin, signal.zmax),
        (ground.zmin, ground.zmax),
        aligned_stack.dielectrics,
        ALIGNED_PROFILE["air_um"],
        ALIGNED_PROFILE["corner_um"],
        ALIGNED_PROFILE["far_um"],
        False,
        False,
        ALIGNED_PROFILE["mirror"],
    )
    for label, geometry, gap in [
        ("intact", GEO, False),
        ("disrupted", GEO, True),
        ("moved", replace(GEO, gap_x=ALIGNED_SLOT_CENTRES[1]), True),
    ]:
        aligned_networks[label] = aligned_run_case(
            label,
            geometry,
            aligned_stack,
            SIGNAL_KEY,
            RETURN_KEY,
            gap=gap,
            profile=ALIGNED_PROFILE,
            overrides=MATERIAL_OVERRIDES,
            section=shared_section,
            slot_centres=ALIGNED_SLOT_CENTRES,
            dense=ALIGNED_DENSE,
            anchors_ghz=ALIGNED_ANCHORS_GHZ,
            provenance={"pdk_source_revision": IHP_REVISION},
        )
else:
    print(
        "Aligned full-network study INCOMPLETE: RUN_ALIGNED=False; pilot columns are not substituted."
    )

### 16.6 Check separation from PDK names

The optional setup fixture changes layer/material names and heights and uses a
single finite dielectric. This is a geometry/configuration check without an EM
field solve.

In [ ]:
# A cheap generic setup check; no electromagnetic field solve is launched.
CHECK_ALIGNED_SETUP = False
if CHECK_ALIGNED_SETUP:
    fixture_stack = build_palace_layer_stack(dummy_stack)
    for layer in fixture_stack.layers.values():
        layer.material = "fixture_metal"
    fixture_stack.materials = {
        "fixture_metal": {"conductivity": 3e7},
        "fixture_glass": {"permittivity": 3.2, "loss_tangent": 0.002},
    }
    fixture_stack.dielectrics = [
        {"name": "bulk", "material": "fixture_glass", "zmin": 0.0, "zmax": 11.0}
    ]
    fixture_geometry = replace(GEO, length=200.0, gap_x=100.0, gap_length=20.0)
    fixture_profile = {
        **ALIGNED_PROFILE,
        "air_um": 30.0,
        "axial_um": 40.0,
        "far_um": 20.0,
    }
    fixture_mesh, fixture_config, fixture_summary = aligned_config(
        fixture_stack,
        asdict(fixture_geometry),
        "trace_sheet",
        "return_sheet",
        OUTPUT / "aligned-synthetic-setup",
        profile=fixture_profile,
        gap=True,
        slot_centres=(100.0, 140.0),
    )
    assert set(fixture_summary["material_ids"]) == {"air", "fixture_glass"}
    assert set(fixture_summary["groups"]["conductor_surfaces"]) == {
        "trace_sheet_xy",
        "trace_sheet_z",
        "return_sheet_xy",
        "return_sheet_z",
    }
    assert all(p["zmin"] == 2.5 and p["zmax"] == 9.0 for p in fixture_mesh.port_info)
    validate_generated_config(fixture_config)
    print(
        "Aligned geometry/configuration accepted different layer/material names and heights."
    )